In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Imports and helper functions
# ─────────────────────────────────────────────────────────────────────────────
import re
from pathlib import Path

import pandas as pd
import numpy as np
import requests

pd.set_option('display.max_columns', None)


def readLocalTable(path):
    """Read a local parquet/CSV table, falling back to CSV beside parquet."""
    path = Path(path)
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)

    if path.suffix.lower() == '.parquet':
        try:
            return pd.read_parquet(path)
        except ImportError as exc:
            csvPath = path.with_suffix('.csv')
            if csvPath.exists():
                return pd.read_csv(csvPath)
            raise ImportError(
                f'Cannot read {path} because pandas needs pyarrow or fastparquet '
                f'for parquet files, and no CSV fallback exists at {csvPath}.'
            ) from exc

    raise ValueError(f'Unsupported table file type: {path.suffix}')


def _modeOrNA(values):
    values = values.dropna()
    if values.empty:
        return pd.NA
    return values.mode().iloc[0]


def _seasonFromGameId(gameIdValue):
    gameIdText = str(gameIdValue)
    try:
        seasonStart = 2000 + int(gameIdText[3:5])
    except (TypeError, ValueError):
        return pd.NA
    return f'{seasonStart}-{str(seasonStart + 1)[-2:]}'


def inferGameMetaFromPlayByPlay(pbpDf):
    """Infer game-level metadata from Live API scoring rows.

    This is a fallback for environments without a parquet engine. The Live API
    action rows include scoreHome/scoreAway and the scoring team; score deltas
    identify which team is home or away.
    """
    df = pbpDf.copy()
    df['gameId'] = df['gameId'].astype(str)
    df['teamIdNumeric'] = pd.to_numeric(df.get('teamId'), errors='coerce')

    scoreCols = ['scoreHome', 'scoreAway']
    for scoreCol in scoreCols:
        df[scoreCol] = pd.to_numeric(df[scoreCol], errors='coerce')
        df[scoreCol] = df.groupby('gameId')[scoreCol].ffill().fillna(0)

    sortCols = ['gameId'] + (['actionNumber'] if 'actionNumber' in df.columns else [])
    df = df.sort_values(sortCols).reset_index(drop=True)
    scoreDiffs = df.groupby('gameId')[scoreCols].diff().fillna(df[scoreCols])
    df['homeScoreDelta'] = scoreDiffs['scoreHome']
    df['awayScoreDelta'] = scoreDiffs['scoreAway']

    scoringRows = df.loc[
        df['teamIdNumeric'].notna()
        & (df['homeScoreDelta'].gt(0) ^ df['awayScoreDelta'].gt(0))
    ].copy()

    homeRows = scoringRows.loc[scoringRows['homeScoreDelta'].gt(0)]
    awayRows = scoringRows.loc[scoringRows['awayScoreDelta'].gt(0)]

    homeTeams = homeRows.groupby('gameId').agg(
        homeTeamId=('teamIdNumeric', lambda s: int(_modeOrNA(s))),
        homeAbbreviation=('teamTricode', _modeOrNA),
    )
    awayTeams = awayRows.groupby('gameId').agg(
        awayTeamId=('teamIdNumeric', lambda s: int(_modeOrNA(s))),
        awayAbbreviation=('teamTricode', _modeOrNA),
    )

    finalScores = df.groupby('gameId')[scoreCols].last()
    firstTimes = pd.to_datetime(
        df.groupby('gameId')['timeActual'].first(), errors='coerce', utc=True
    )
    gameDates = (firstTimes - pd.Timedelta(hours=12)).dt.date

    metaDf = (
        pd.DataFrame(index=pd.Index(df['gameId'].unique(), name='gameId'))
        .join(homeTeams)
        .join(awayTeams)
        .join(finalScores.rename(columns={'scoreHome': 'finalHomeScore', 'scoreAway': 'finalAwayScore'}))
    )
    metaDf['season'] = metaDf.index.map(_seasonFromGameId)
    metaDf['gameDate'] = gameDates.reindex(metaDf.index)
    metaDf['matchup'] = metaDf['homeAbbreviation'].astype('string') + ' vs. ' + metaDf['awayAbbreviation'].astype('string')
    metaDf['homeWin'] = metaDf['finalHomeScore'].gt(metaDf['finalAwayScore']).astype('Int64')

    return metaDf.reset_index()[[
        'season', 'gameId', 'gameDate', 'matchup',
        'homeTeamId', 'homeAbbreviation', 'awayTeamId', 'awayAbbreviation', 'homeWin'
    ]]


def clockToSeconds(clockValue):
    """Convert 'PT12M30.00S' NBA clock string to seconds left in the period."""
    if pd.isna(clockValue) or str(clockValue).strip() == '':
        return pd.NA
    clockText = str(clockValue).replace('PT', '').replace('S', '')
    minutesText, secondsText = clockText.split('M', 1)
    return int(minutesText) * 60 + float(secondsText)


def secondsLeftInGame(periodValue, clockValue, periodTypeValue='REGULAR'):
    """Total seconds remaining in the game at the moment of an action.

    Regulation: 4 quarters × 720 s each.
    Overtime:   5-min periods (300 s); period 5 = OT1, period 6 = OT2, …
    """
    clockSeconds = clockToSeconds(clockValue)
    if str(periodTypeValue).upper() == 'OVERTIME' or periodValue > 4:
        overtimeNumber = max(periodValue - 5, 0)
        return overtimeNumber * 300 + clockSeconds
    return max(4 - periodValue, 0) * 720 + clockSeconds


def loadAllGameMeta(gameIds, pbpDf=None):
    """Return a DataFrame of home/away metadata for every gameId in *gameIds*.

    Prefers local gamelog files. If parquet support is unavailable, falls back
    to metadata inferred from the play-by-play scoring rows.
    """
    candidateDirs = [Path('nba_gamelog'), Path('data/nba_gamelog')]
    gameIdSet = {str(g) for g in gameIds}
    readErrors = []

    for candidateDir in candidateDirs:
        paths = sorted(candidateDir.glob('gamelog_*.parquet'), reverse=True)
        if not paths:
            continue

        try:
            gamelogDf = pd.concat([readLocalTable(p) for p in paths], ignore_index=True)
        except ImportError as exc:
            readErrors.append(str(exc))
            continue

        gamelogDf.columns = gamelogDf.columns.str.lower().str.replace(
            r'_(\w)', lambda m: m.group(1).upper(), regex=True
        )
        gamelogDf['gameId'] = gamelogDf['gameId'].astype(str)
        gamelogDf = gamelogDf.loc[gamelogDf['gameId'].isin(gameIdSet)]

        if gamelogDf.empty:
            continue

        # Home rows: 'BOS vs. PHI'  |  Away rows: 'PHI @ BOS'
        homeRows = (
            gamelogDf
            .loc[gamelogDf['matchup'].str.contains(' vs. ', na=False),
                 ['gameId', 'season', 'gameDate', 'matchup', 'teamId', 'teamAbbreviation', 'wl']]
            .copy()
            .rename(columns={'teamId': 'homeTeamId', 'teamAbbreviation': 'homeAbbreviation'})
        )
        awayRows = (
            gamelogDf
            .loc[gamelogDf['matchup'].str.contains(' @ ', na=False),
                 ['gameId', 'teamId', 'teamAbbreviation']]
            .copy()
            .rename(columns={'teamId': 'awayTeamId', 'teamAbbreviation': 'awayAbbreviation'})
        )

        metaDf = homeRows.merge(awayRows, on='gameId', how='left')
        metaDf['gameDate'] = pd.to_datetime(metaDf['gameDate']).dt.date
        metaDf['homeWin']  = metaDf['wl'].map({'W': 1, 'L': 0})
        return metaDf.drop(columns=['wl'])

    if pbpDf is not None:
        return inferGameMetaFromPlayByPlay(pbpDf).loc[lambda d: d['gameId'].isin(gameIdSet)]

    if readErrors:
        raise ImportError('Could not read local gamelog parquet files. ' + ' '.join(readErrors))
    raise FileNotFoundError(f'No local gamelog found in {[str(d) for d in candidateDirs]}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load all available play-by-play files under data/nba_pbp_21-26/data
# gameId is not embedded in the action data, so it is read from the filename.
# ─────────────────────────────────────────────────────────────────────────────
# This path is probably stale if anyone runs this again
dataRoot = Path('nba_pbp_21-26/data')
seasonDirs = [
    dataRoot / '2021-22',
    dataRoot / '2022-23',
    dataRoot / '2023-24',
    dataRoot / '2024-25',
    dataRoot / '2025-26',
]

# CSV-only on purpose: the parquet files in this directory use a different actionType / subType
# taxonomy than the CSVs.
csvFiles = []
for seasonDir in seasonDirs:
    if seasonDir.exists():
        csvFiles.extend(sorted(seasonDir.rglob('*.csv')))

byGame = {}
for p in csvFiles:
    byGame.setdefault(p.stem, p)

dataFiles = [byGame[k] for k in sorted(byGame)]
print(f'Found {len(csvFiles)} CSV files ({len(byGame)} unique games)')

if not dataFiles:
    raise FileNotFoundError(f'No play-by-play files found in season folders under {dataRoot}')

parts = []
for p in dataFiles:
    gameDf = readLocalTable(p)
    gameDf['gameId'] = p.stem   # filename stem = NBA game ID
    # Sort by actionNumber so events are in canonical chronological order.
    # The live API occasionally delivers rows out of order around period
    # transitions (e.g. period-start before end-of-period free throws).
    if 'actionNumber' in gameDf.columns:
        gameDf = gameDf.sort_values('actionNumber').reset_index(drop=True)
    parts.append(gameDf)

pbp_all = pd.concat(parts, ignore_index=True)
print(f'Total rows : {len(pbp_all):,}')
print(f'Games      : {pbp_all["gameId"].nunique()}')
pbp_all.head(3)


Found 6140 CSV files (6140 unique games)
Total rows : 3,433,823
Games      : 6140


,actionNumber,clock,timeActual,period,periodType,actionType,subType,qualifiers,personId,x,y,possession,scoreHome,scoreAway,edited,orderNumber,xLegacy,yLegacy,isFieldGoal,side,description,personIdsFilter,teamId,teamTricode,descriptor,jumpBallRecoveredName,jumpBallRecoverdPersonId,playerName,playerNameI,jumpBallWonPlayerName,jumpBallWonPersonId,jumpBallLostPlayerName,jumpBallLostPersonId,shotDistance,shotResult,shotActionNumber,reboundTotal,reboundDefensiveTotal,reboundOffensiveTotal,officialId,foulPersonalTotal,foulTechnicalTotal,foulDrawnPlayerName,foulDrawnPersonId,pointsTotal,assistPlayerNameInitial,assistPersonId,assistTotal,blockPlayerName,blockPersonId,turnoverTotal,stealPlayerName,stealPersonId,value,gameId,area,areaDetail,isTargetScoreLastPeriod,shortFormattedClock,jerseyNumber,location,videoAvailable,shotValue,actionId
0,2,PT12M00.00S,2021-10-19T23:37:02.0Z,1,REGULAR,period,start,[],0,NaN,NaN,0.000000e+00,0.0,0.0,2021-10-19T23:37:02Z,20000.0,NaN,NaN,0,NaN,Period Start,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0022100001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,PT11M55.00S,2021-10-19T23:37:06.1Z,1,REGULAR,jumpball,recovered,[],203507,NaN,NaN,1.610613e+09,0.0,0.0,2021-10-19T23:37:06Z,40000.0,NaN,NaN,0,NaN,Jump Ball B. Lopez vs. N. Claxton: Tip to G. A...,"[203507, 201572, 1629651]",1.610613e+09,MIL,startperiod,G. Antetokounmpo,203507.0,Antetokounmpo,G. Antetokounmpo,Lopez,201572.0,Claxton,1629651.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0022100001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7,PT11M42.00S,2021-10-19T23:37:18.4Z,1,REGULAR,3pt,Jump Shot,[],1628960,27.94021,82.843137,1.610613e+09,0.0,0.0,2021-10-19T23:37:24Z,70000.0,-164.0,210.0,1,left,MISS G. Allen 26' 3PT,[1628960],1.610613e+09,MIL,NaN,NaN,NaN,Allen,G. Allen,NaN,NaN,NaN,NaN,26.67,Missed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0022100001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Normalize actionType values to canonical labels.
if 'actionType' in pbp_all.columns:
    action_type_map = {
        'rebound': 'rebound',
        'missed shot': 'missed shot',
        'made shot': 'made shot',
        'substitution': 'substitution',
        'free throw': 'freethrow',
        'freethrow': 'freethrow',
        'foul': 'foul',
        'turnover': 'turnover',
        'timeout': 'timeout',
        'period': 'period',
        'instant replay': 'instant replay',
        'violation': 'violation',
        'jump ball': 'jumpball',
        'jumpball': 'jumpball',
        '2pt': '2pt',
        '3pt': '3pt',
        'heave': 'heave',
        'ejection': 'ejection',
        'steal': 'steal',
        'block': 'block',
        'game': 'game',
        '': '',
    }

    normalized = (
        pbp_all['actionType']
        .astype('string')
        .str.strip()
        .str.lower()
        .map(action_type_map)
    )
    pbp_all['actionType'] = normalized.fillna(pbp_all['actionType']).astype('string')

pbp_all['actionType'].value_counts(dropna=False).head(30)

actionType
substitution      693462
2pt               648253
rebound           639157
3pt               438754
freethrow         275687
foul              245182
turnover          172553
steal              95569
timeout            67553
block              59346
period             49818
stoppage           14103
violation          10826
jumpball           10790
game                6127
instantreplay       2062
missed shot         1295
made shot           1197
heave               1093
ejection             410
<NA>                 370
memo                 186
instant replay        30
Name: count, dtype: Int64

In [4]:

# ─────────────────────────────────────────────────────────────────────────────
# Merge game-level metadata and compute all derived columns
# ─────────────────────────────────────────────────────────────────────────────

# --- Game meta: season, date, matchup, home/away team IDs, win result ----------
metaDf = loadAllGameMeta(pbp_all['gameId'].unique(), pbp_all)
pbp_all = pbp_all.merge(metaDf, on='gameId', how='left')

# --- Scores (arrive as strings from the live API; ffill within each game) ------
# Empty/NaN entries between scoring plays are forward-filled with the last score.
for scoreCol in ['scoreHome', 'scoreAway']:
    if scoreCol in pbp_all.columns:
        pbp_all[scoreCol] = (
            pbp_all
            .groupby('gameId')[scoreCol]
            .transform(lambda s: pd.to_numeric(s, errors='coerce').ffill().fillna(0))
        )

pbp_all['pointsTotal'] = pbp_all['scoreHome'] + pbp_all['scoreAway']
pbp_all['scoreDif']    = pbp_all['scoreHome'] - pbp_all['scoreAway']

# --- Period labels and clocks -------------------------------------------------
pbp_all['periodNumber'] = pd.to_numeric(
    pbp_all['period'], errors='coerce'
).astype('Int64')

# quarter: compact period label for model features and display.
# Regular periods -> 'Q1'...'Q4'; overtime periods stay grouped as 'OT'.
pbp_all['quarter'] = pbp_all['periodNumber'].apply(
    lambda p: f'Q{int(p)}' if pd.notna(p) and int(p) <= 4 else 'OT'
)

# periodSecondsLeft: clock remaining in the current period.
pbp_all['periodSecondsLeft'] = pbp_all['clock'].apply(clockToSeconds)

# secondsLeft: total regulation/OT game-clock proxy at each action.
pbp_all['secondsLeft'] = pbp_all.apply(
    lambda row: secondsLeftInGame(
        row['period'],
        row['clock'],
        row.get('periodType', 'REGULAR')
    ),
    axis=1
)

# --- actionTeamSide: which side (home/away) committed this action --------------
if 'teamId' in pbp_all.columns:
    pbp_all['teamId']     = pd.to_numeric(pbp_all['teamId'],     errors='coerce').astype('Int64')
    pbp_all['homeTeamId'] = pd.to_numeric(pbp_all['homeTeamId'], errors='coerce').astype('Int64')
    pbp_all['awayTeamId'] = pd.to_numeric(pbp_all['awayTeamId'], errors='coerce').astype('Int64')

    pbp_all['isHomeAction']   = pbp_all['teamId'].eq(pbp_all['homeTeamId'])
    pbp_all.loc[pbp_all['teamId'].isna(), 'isHomeAction'] = pd.NA

    pbp_all['actionTeamSide'] = pd.Series(pd.NA, index=pbp_all.index, dtype='object')
    pbp_all.loc[pbp_all['teamId'].eq(pbp_all['homeTeamId']), 'actionTeamSide'] = 'home'
    pbp_all.loc[pbp_all['teamId'].eq(pbp_all['awayTeamId']), 'actionTeamSide'] = 'away'

# --- possessionTeamSide: which side currently has the ball --------------------
if 'possession' in pbp_all.columns:
    pbp_all['possession'] = pd.to_numeric(pbp_all['possession'], errors='coerce').astype('Int64')

    pbp_all['isHomePossession']   = pbp_all['possession'].eq(pbp_all['homeTeamId'])
    pbp_all.loc[pbp_all['possession'].isna(), 'isHomePossession'] = pd.NA

    pbp_all['possessionTeamSide'] = pd.Series(pd.NA, index=pbp_all.index, dtype='object')
    pbp_all.loc[pbp_all['possession'].eq(pbp_all['homeTeamId']), 'possessionTeamSide'] = 'home'
    pbp_all.loc[pbp_all['possession'].eq(pbp_all['awayTeamId']), 'possessionTeamSide'] = 'away'

# --- Reorder: game identifiers first, then action details ---------------------
preferredColumns = [
    'season', 'gameId', 'gameDate', 'matchup',
    'homeTeamId', 'homeAbbreviation', 'awayTeamId', 'awayAbbreviation', 'homeWin',
    'description', 'periodNumber', 'quarter', 'periodSecondsLeft', 'secondsLeft',
    'scoreHome', 'scoreAway', 'scoreDif', 'pointsTotal',
    'actionTeamSide',
    'possession',
    'actionType', 'subType', 'personId', 'playerName',
    'shotResult', 'foulPersonalTotal', 'foulTechnicalTotal',
]
existingCols  = [c for c in preferredColumns if c in pbp_all.columns]
remainingCols = [c for c in pbp_all.columns if c not in existingCols]
pbp_all = pbp_all[existingCols + remainingCols]

pbp_all.head(3)


,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,periodNumber,quarter,periodSecondsLeft,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,actionType,subType,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,actionNumber,clock,timeActual,period,periodType,qualifiers,x,y,edited,orderNumber,xLegacy,yLegacy,isFieldGoal,side,personIdsFilter,teamId,teamTricode,descriptor,jumpBallRecoveredName,jumpBallRecoverdPersonId,playerNameI,jumpBallWonPlayerName,jumpBallWonPersonId,jumpBallLostPlayerName,jumpBallLostPersonId,shotDistance,shotActionNumber,reboundTotal,reboundDefensiveTotal,reboundOffensiveTotal,officialId,foulDrawnPlayerName,foulDrawnPersonId,assistPlayerNameInitial,assistPersonId,assistTotal,blockPlayerName,blockPersonId,turnoverTotal,stealPlayerName,stealPersonId,value,area,areaDetail,isTargetScoreLastPeriod,shortFormattedClock,jerseyNumber,location,videoAvailable,shotValue,actionId,isHomeAction,isHomePossession,possessionTeamSide
0,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Period Start,1,Q1,720.0,2880.0,0.0,0.0,0.0,0.0,<NA>,0,period,start,0,NaN,NaN,NaN,NaN,2,PT12M00.00S,2021-10-19T23:37:02.0Z,1,REGULAR,[],NaN,NaN,2021-10-19T23:37:02Z,20000.0,NaN,NaN,0,NaN,[],<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,False,<NA>
1,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Jump Ball B. Lopez vs. N. Claxton: Tip to G. A...,1,Q1,715.0,2875.0,0.0,0.0,0.0,0.0,home,1610612749,jumpball,recovered,203507,Antetokounmpo,NaN,NaN,NaN,4,PT11M55.00S,2021-10-19T23:37:06.1Z,1,REGULAR,[],NaN,NaN,2021-10-19T23:37:06Z,40000.0,NaN,NaN,0,NaN,"[203507, 201572, 1629651]",1610612749,MIL,startperiod,G. Antetokounmpo,203507.0,G. Antetokounmpo,Lopez,201572.0,Claxton,1629651.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True,home
2,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,MISS G. Allen 26' 3PT,1,Q1,702.0,2862.0,0.0,0.0,0.0,0.0,home,1610612749,3pt,Jump Shot,1628960,Allen,Missed,NaN,NaN,7,PT11M42.00S,2021-10-19T23:37:18.4Z,1,REGULAR,[],27.94021,82.843137,2021-10-19T23:37:24Z,70000.0,-164.0,210.0,1,left,[1628960],1610612749,MIL,NaN,NaN,NaN,G. Allen,NaN,NaN,NaN,NaN,26.67,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True,home


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Drop raw/granular columns not needed downstream
# ─────────────────────────────────────────────────────────────────────────────
colsToDrop = [
    # superseded by derived columns
    'isHomePossession', 'isHomeAction', 'teamId', 'teamTricode', 'period',
    'actionNumber', 'clock', 'timeActual', 'periodType',
    # positional / shot-chart
    'x', 'y', 'xLegacy', 'yLegacy', 'shotDistance', 'isFieldGoal', 'area', 'areaDetail',
    # misc flags & filters
    'edited', 'orderNumber', 'qualifiers', 'side', 'personIdsFilter', 'descriptor',
    'isTargetScoreLastPeriod',
    # jump ball detail
    'jumpBallRecoveredName', 'jumpBallRecoverdPersonId', 'playerNameI',
    'jumpBallWonPlayerName', 'jumpBallWonPersonId',
    'jumpBallLostPlayerName', 'jumpBallLostPersonId',
    # shot / rebound / assist detail
    'shotActionNumber', 'reboundTotal', 'reboundDefensiveTotal', 'reboundOffensiveTotal',
    'blockPlayerName', 'blockPersonId',
    'assistPlayerNameInitial', 'assistPersonId', 'assistTotal',
    # foul / turnover / steal detail
    'foulDrawnPlayerName', 'foulDrawnPersonId',
    'stealPlayerName', 'stealPersonId',
    'officialId', 'turnoverTotal',
]
colsToDrop = [c for c in colsToDrop if c in pbp_all.columns]
pbp_all = pbp_all.drop(columns=colsToDrop)

print(pbp_all.columns.tolist())

['season', 'gameId', 'gameDate', 'matchup', 'homeTeamId', 'homeAbbreviation', 'awayTeamId', 'awayAbbreviation', 'homeWin', 'description', 'periodNumber', 'quarter', 'periodSecondsLeft', 'secondsLeft', 'scoreHome', 'scoreAway', 'scoreDif', 'pointsTotal', 'actionTeamSide', 'possession', 'actionType', 'subType', 'personId', 'playerName', 'shotResult', 'foulPersonalTotal', 'foulTechnicalTotal', 'value', 'shortFormattedClock', 'jerseyNumber', 'location', 'videoAvailable', 'shotValue', 'actionId', 'possessionTeamSide']


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Merge Vegas betting lines from Rotowire
# Matched on gameDate × homeAbbreviation × awayAbbreviation.
# If the request fails the rest of the pipeline continues without the 'line' column.
# ─────────────────────────────────────────────────────────────────────────────
rotowireUrl = 'https://www.rotowire.com/betting/nba/tables/games-archive.php'
try:
    resp = requests.get(rotowireUrl, timeout=30)
    resp.raise_for_status()
    rotowireDf = (
        pd.DataFrame(resp.json())
        [['game_date', 'home_team_abbrev', 'visit_team_abbrev', 'line']]
        .rename(columns={
            'game_date':         'gameDate',
            'home_team_abbrev':  'homeAbbreviation',
            'visit_team_abbrev': 'awayAbbreviation',
        })
    )
    rotowireDf['gameDate'] = pd.to_datetime(rotowireDf['gameDate']).dt.date
    rotowireDf = rotowireDf.drop_duplicates(subset=['gameDate', 'homeAbbreviation', 'awayAbbreviation'])
    pbp_all = pbp_all.merge(
        rotowireDf, on=['gameDate', 'homeAbbreviation', 'awayAbbreviation'], how='left'
    )
    print('Rotowire merge successful')
except Exception as exc:
    print(f'Rotowire merge skipped: {exc}')

pbp_all.head(3)

Rotowire merge successful


,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,periodNumber,quarter,periodSecondsLeft,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,actionType,subType,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,value,shortFormattedClock,jerseyNumber,location,videoAvailable,shotValue,actionId,possessionTeamSide,line
0,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Period Start,1,Q1,720.0,2880.0,0.0,0.0,0.0,0.0,<NA>,0,period,start,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,1.5
1,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Jump Ball B. Lopez vs. N. Claxton: Tip to G. A...,1,Q1,715.0,2875.0,0.0,0.0,0.0,0.0,home,1610612749,jumpball,recovered,203507,Antetokounmpo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home,1.5
2,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,MISS G. Allen 26' 3PT,1,Q1,702.0,2862.0,0.0,0.0,0.0,0.0,home,1610612749,3pt,Jump Shot,1628960,Allen,Missed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home,1.5


In [7]:

# ─────────────────────────────────────────────────────────────────────────────
# Per-team, per-period foul counts and bonus indicators
# ─────────────────────────────────────────────────────────────────────────────

bonus_columns = [
    'homeFouls', 'awayFouls',
    'homeFoulsAtCheckpoint', 'awayFoulsAtCheckpoint',
    'homeNonPenaltyFoulLimit', 'awayNonPenaltyFoulLimit',
    'homeBonus', 'awayBonus',
]


def _teamFoulMask(pbp_df):
    """Rows that count toward NBA team-foul bonus state."""
    subtype_lower = pbp_df.get('subType', pd.Series('', index=pbp_df.index)).fillna('').str.lower()
    description_lower = pbp_df.get('description', pd.Series('', index=pbp_df.index)).fillna('').str.lower()

    return (
        pbp_df['actionType'].eq('foul') &
        pbp_df['actionTeamSide'].isin(['home', 'away']) &
        ~subtype_lower.str.contains('technical|offensive|double', regex=True) &
        ~description_lower.str.contains('double', regex=False)
    )


def applyTeamFoulBonusRules(pbp_df):
    """Add NBA team-foul counts, checkpoint counts, limits, and bonus flags.

    Rule shape:
    - Regulation: first 4 team fouls in a period are non-penalty.
    - Overtime: first 3 team fouls in an OT period are non-penalty.
    - Final 2:00: bonus threshold is whichever comes first between
      one foul in the segment and the period quota (reg 4 / OT 3),
      i.e. min(quota, fouls_before_final_2min + 1).
    """
    result = pbp_df.drop(columns=bonus_columns, errors='ignore').copy()

    team_foul_mask = _teamFoulMask(result)
    foul_df = result.loc[
        team_foul_mask,
        ['gameId', 'periodNumber', 'actionTeamSide', 'periodSecondsLeft', 'personId', 'playerName', 'foulPersonalTotal']
    ].copy()
    foul_df = (
        foul_df
        .reset_index(names='foulIndex')
        .rename(columns={'actionTeamSide': 'playerHomeAway'})
    )

    # For each action, count how many team fouls each side has committed within
    # the same game x period up to and including that row. The row-index tie
    # breaker keeps same-clock events in action order instead of counting future
    # rows at that same clock.
    df_idx = result[['gameId', 'periodNumber', 'periodSecondsLeft']].reset_index()

    def countFoulsThroughAction(action_df, foul_events):
        if foul_events.empty:
            return pd.Series(0, index=action_df['index'], dtype='int64')

        merged = action_df.merge(
            foul_events[['gameId', 'periodNumber', 'periodSecondsLeft', 'foulIndex']],
            on=['gameId', 'periodNumber'],
            suffixes=('', '_foul')
        )
        happened_by_action = (
            (merged['periodSecondsLeft_foul'] > merged['periodSecondsLeft']) |
            (
                merged['periodSecondsLeft_foul'].eq(merged['periodSecondsLeft']) &
                merged['foulIndex'].le(merged['index'])
            )
        )
        counts = merged.loc[happened_by_action].groupby('index').size()
        return counts.reindex(action_df['index'], fill_value=0).astype('int64')

    home_foul_events = foul_df[foul_df['playerHomeAway'] == 'home']
    away_foul_events = foul_df[foul_df['playerHomeAway'] == 'away']

    result['homeFouls'] = countFoulsThroughAction(df_idx, home_foul_events).to_numpy()
    result['awayFouls'] = countFoulsThroughAction(df_idx, away_foul_events).to_numpy()
    home_foul_events_last_two = home_foul_events[home_foul_events['periodSecondsLeft'].le(120)]
    away_foul_events_last_two = away_foul_events[away_foul_events['periodSecondsLeft'].le(120)]
    home_fouls_last_two = countFoulsThroughAction(df_idx, home_foul_events_last_two).to_numpy()
    away_fouls_last_two = countFoulsThroughAction(df_idx, away_foul_events_last_two).to_numpy()

    # A foul at exactly 2:00 belongs to the final-two-minute segment, so
    # checkpoint counts include only fouls with periodSecondsLeft > 120.
    def checkpointCounts(foul_events, column_name):
        counts = (
            foul_events[foul_events['periodSecondsLeft'] > 120]
            .groupby(['gameId', 'periodNumber'])
            .size()
            .rename(column_name)
            .reset_index()
        )
        return counts

    result = result.merge(
        checkpointCounts(home_foul_events, 'homeFoulsAtCheckpoint'),
        on=['gameId', 'periodNumber'],
        how='left'
    )
    result = result.merge(
        checkpointCounts(away_foul_events, 'awayFoulsAtCheckpoint'),
        on=['gameId', 'periodNumber'],
        how='left'
    )

    home_checkpoint_all = result['homeFoulsAtCheckpoint'].fillna(0).astype('int64')
    away_checkpoint_all = result['awayFoulsAtCheckpoint'].fillna(0).astype('int64')

    in_last_two = result['periodSecondsLeft'].le(120).fillna(False)
    is_regulation = result['periodNumber'].le(4).fillna(True)
    team_foul_quota = pd.Series(4, index=result.index).where(is_regulation, 3).astype('int64')

    def buildNonPenaltyLimit(checkpoint_counts):
        late_limit = checkpoint_counts + 1
        late_limit = late_limit.where(late_limit.le(team_foul_quota), team_foul_quota)
        return team_foul_quota.where(~in_last_two, late_limit).astype('int64')

    result['homeNonPenaltyFoulLimit'] = buildNonPenaltyLimit(home_checkpoint_all)
    result['awayNonPenaltyFoulLimit'] = buildNonPenaltyLimit(away_checkpoint_all)

    # Only expose checkpoint counts once the checkpoint has actually been reached;
    # before then, exposing the final checkpoint count would leak future fouls.
    result['homeFoulsAtCheckpoint'] = home_checkpoint_all
    result['awayFoulsAtCheckpoint'] = away_checkpoint_all
    result.loc[~in_last_two, ['homeFoulsAtCheckpoint', 'awayFoulsAtCheckpoint']] = pd.NA
    result[['homeFoulsAtCheckpoint', 'awayFoulsAtCheckpoint']] = (
        result[['homeFoulsAtCheckpoint', 'awayFoulsAtCheckpoint']]
        .astype('Int64')
    )

    # homeBonus / awayBonus are based on separate counters:
    # - overall fouls in period reaching quota (reg 4 / OT 3), OR
    # - at least one qualifying foul committed in final 2:00.
    away_reached_quota = result['awayFouls'] >= team_foul_quota
    home_reached_quota = result['homeFouls'] >= team_foul_quota
    away_foul_in_last_two = away_fouls_last_two >= 1
    home_foul_in_last_two = home_fouls_last_two >= 1
    result['homeBonus'] = (away_reached_quota | away_foul_in_last_two).astype(int)
    result['awayBonus'] = (home_reached_quota | home_foul_in_last_two).astype(int)

    return result


pbp_all = applyTeamFoulBonusRules(pbp_all)

pbp_all[[
    'gameId', 'periodNumber', 'quarter', 'periodSecondsLeft', 'secondsLeft',
    'homeFouls', 'awayFouls',
    'homeFoulsAtCheckpoint', 'awayFoulsAtCheckpoint',
    'homeNonPenaltyFoulLimit', 'awayNonPenaltyFoulLimit',
    'homeBonus', 'awayBonus'
]].head(20)


,gameId,periodNumber,quarter,periodSecondsLeft,secondsLeft,homeFouls,awayFouls,homeFoulsAtCheckpoint,awayFoulsAtCheckpoint,homeNonPenaltyFoulLimit,awayNonPenaltyFoulLimit,homeBonus,awayBonus
0,0022100001,1,Q1,720.0,2880.0,0,0,<NA>,<NA>,4,4,0,0
1,0022100001,1,Q1,715.0,2875.0,0,0,<NA>,<NA>,4,4,0,0
2,0022100001,1,Q1,702.0,2862.0,0,0,<NA>,<NA>,4,4,0,0
3,0022100001,1,Q1,699.0,2859.0,0,0,<NA>,<NA>,4,4,0,0
4,0022100001,1,Q1,687.0,2847.0,1,0,<NA>,<NA>,4,4,0,0
5,0022100001,1,Q1,687.0,2847.0,1,0,<NA>,<NA>,4,4,0,0
6,0022100001,1,Q1,687.0,2847.0,1,0,<NA>,<NA>,4,4,0,0
7,0022100001,1,Q1,687.0,2847.0,1,0,<NA>,<NA>,4,4,0,0
8,0022100001,1,Q1,685.0,2845.0,1,0,<NA>,<NA>,4,4,0,0
9,0022100001,1,Q1,673.0,2833.0,1,0,<NA>,<NA>,4,4,0,0


In [8]:

# ─────────────────────────────────────────────────────────────────────────────
# Test: NBA team foul / bonus rule
# ─────────────────────────────────────────────────────────────────────────────

print('=' * 80)
print('TEST: NBA Team Foul / Bonus Rule')
print('=' * 80)


def _bonus_test_rows(events, game_id='test', period_number=1, quarter='Q1'):
    rows = []
    for i, event in enumerate(events):
        row = {
            'gameId': game_id,
            'periodNumber': period_number,
            'quarter': quarter,
            'periodSecondsLeft': event.get('periodSecondsLeft', 600.0 - i),
            'secondsLeft': event.get('secondsLeft', event.get('periodSecondsLeft', 600.0 - i)),
            'description': event.get('description', ''),
            'actionType': event.get('actionType', 'foul'),
            'subType': event.get('subType', 'personal'),
            'actionTeamSide': event.get('actionTeamSide', 'home'),
            'personId': event.get('personId', 1),
            'playerName': event.get('playerName', 'Test'),
            'foulPersonalTotal': event.get('foulPersonalTotal', 1),
        }
        rows.append(row)
    return rows


def _apply_bonus_test(events, game_id='test', period_number=1, quarter='Q1'):
    return applyTeamFoulBonusRules(pd.DataFrame(
        _bonus_test_rows(events, game_id, period_number, quarter)
    ))

# Regulation examples: penalty begins on limit + 1 because the current row's
# team-foul count is inclusive.
reg_examples = pd.DataFrame({'foulsAt2Min': [0, 1, 2, 3, 4, 5]})
reg_examples['nonPenaltyFoulLimit'] = (reg_examples['foulsAt2Min'] + 1).clip(upper=4)
reg_examples['penaltyStartsOn'] = reg_examples['nonPenaltyFoulLimit'] + 1
print('\nRegulation final-2:00 examples')
print(reg_examples)

ot_examples = pd.DataFrame({'foulsAt2Min': [0, 1, 2, 3, 4]})
ot_examples['nonPenaltyFoulLimit'] = (ot_examples['foulsAt2Min'] + 1).clip(upper=3)
ot_examples['penaltyStartsOn'] = ot_examples['nonPenaltyFoulLimit'] + 1
print('\nOvertime final-2:00 examples')
print(ot_examples)

# 0 fouls before final 2:00: one late foul is allowed; second late foul is bonus.
case = _apply_bonus_test([
    {'periodSecondsLeft': 119.0, 'actionTeamSide': 'away'},
    {'periodSecondsLeft': 90.0, 'actionTeamSide': 'away'},
])
print('\nCase: away fouls in final-2 (check homeBonus)')
print(case[['periodSecondsLeft', 'awayFouls', 'awayNonPenaltyFoulLimit', 'homeBonus']])
assert case['awayNonPenaltyFoulLimit'].tolist() == [1, 1]
assert case['homeBonus'].tolist() == [1, 1]

# 2 fouls before final 2:00: third foul is the grace foul; fourth is bonus.
case = _apply_bonus_test([
    {'periodSecondsLeft': 600.0, 'actionTeamSide': 'home'},
    {'periodSecondsLeft': 500.0, 'actionTeamSide': 'home'},
    {'periodSecondsLeft': 110.0, 'actionTeamSide': 'home'},
    {'periodSecondsLeft': 100.0, 'actionTeamSide': 'home'},
])
print('\nCase: regulation -> final-2 transition (check awayBonus)')
print(case[['periodSecondsLeft', 'homeFouls', 'homeNonPenaltyFoulLimit', 'awayBonus']])
assert case['homeFouls'].tolist() == [1, 2, 3, 4]
assert case['homeNonPenaltyFoulLimit'].tolist() == [4, 4, 3, 3]
assert case['awayBonus'].tolist() == [0, 0, 1, 1]

# Foul at exactly 2:00 is in the final-two-minute segment, not the checkpoint.
case = _apply_bonus_test([
    {'periodSecondsLeft': 121.0, 'actionTeamSide': 'home'},
    {'periodSecondsLeft': 120.0, 'actionTeamSide': 'home'},
    {'periodSecondsLeft': 119.0, 'actionTeamSide': 'home'},
])
print('\nCase: 2:00 boundary (check awayBonus)')
print(case[['periodSecondsLeft', 'homeFouls', 'homeNonPenaltyFoulLimit', 'awayBonus']])
assert case['homeFoulsAtCheckpoint'].tolist() == [pd.NA, 1, 1]
assert case['homeNonPenaltyFoulLimit'].tolist() == [4, 2, 2]
assert case['awayBonus'].tolist() == [0, 1, 1]

# Overtime quota is 3, and each OT period resets by periodNumber.
ot_case = pd.concat([
    _apply_bonus_test([
        {'periodSecondsLeft': 240.0, 'actionTeamSide': 'home'},
        {'periodSecondsLeft': 110.0, 'actionTeamSide': 'home'},
        {'periodSecondsLeft': 100.0, 'actionTeamSide': 'home'},
    ], game_id='ot-test', period_number=5, quarter='OT'),
    _apply_bonus_test([
        {'periodSecondsLeft': 250.0, 'secondsLeft': 550.0, 'actionTeamSide': 'home'},
    ], game_id='ot-test', period_number=6, quarter='OT'),
], ignore_index=True)
print('\nCase: OT thresholds across periods (check awayBonus)')
print(ot_case[['periodNumber', 'periodSecondsLeft', 'homeFouls', 'homeNonPenaltyFoulLimit', 'awayBonus']])
assert ot_case['homeFouls'].tolist() == [1, 2, 3, 1]
assert ot_case['homeNonPenaltyFoulLimit'].tolist() == [3, 2, 2, 3]
assert ot_case['awayBonus'].tolist() == [0, 1, 1, 0]

# Offensive, technical, and double fouls do not count toward team-foul bonus.
case = _apply_bonus_test([
    {'periodSecondsLeft': 600.0, 'actionTeamSide': 'home', 'subType': 'offensive', 'description': 'offensive foul'},
    {'periodSecondsLeft': 500.0, 'actionTeamSide': 'home', 'subType': 'technical', 'description': 'technical foul'},
    {'periodSecondsLeft': 400.0, 'actionTeamSide': 'home', 'subType': 'personal', 'description': 'double personal foul'},
    {'periodSecondsLeft': 100.0, 'actionTeamSide': 'home', 'subType': 'personal', 'description': 'personal foul'},
])
print('\nCase: excluded foul types + one late personal (check awayBonus)')
print(case[['periodSecondsLeft', 'subType', 'description', 'homeFouls', 'homeNonPenaltyFoulLimit', 'awayBonus']])
assert case['homeFouls'].tolist() == [0, 0, 0, 1]
assert case['homeNonPenaltyFoulLimit'].tolist() == [4, 4, 4, 1]
assert case['awayBonus'].tolist() == [0, 0, 0, 1]

# Same-clock events are counted in action order, not all at once.
case = _apply_bonus_test([
    {'periodSecondsLeft': 100.0, 'actionTeamSide': 'home'},
    {'periodSecondsLeft': 100.0, 'actionType': 'turnover', 'subType': 'bad pass', 'actionTeamSide': 'home'},
    {'periodSecondsLeft': 100.0, 'actionTeamSide': 'home'},
])
print('\nCase: same-clock ordering (check awayBonus)')
print(case[['periodSecondsLeft', 'actionType', 'homeFouls', 'homeNonPenaltyFoulLimit', 'awayBonus']])
assert case['homeFouls'].tolist() == [1, 1, 2]
assert case['awayBonus'].tolist() == [1, 1, 1]

expected_home_bonus = pbp_all['awayFouls'] >= pbp_all['awayNonPenaltyFoulLimit']
expected_away_bonus = pbp_all['homeFouls'] >= pbp_all['homeNonPenaltyFoulLimit']
assert expected_home_bonus.eq(pbp_all['homeBonus'].astype(bool)).all()
assert expected_away_bonus.eq(pbp_all['awayBonus'].astype(bool)).all()
print('\nSynthetic edge cases and bonus flag invariants: PASS')

quota_by_row = pd.Series(4, index=pbp_all.index).where(
    pbp_all['periodNumber'].le(4).fillna(True),
    3
)
late_rows = pbp_all['periodSecondsLeft'].le(120).fillna(False)
grace_rule_rows = pbp_all[
    late_rows &
    (
        (pbp_all['homeNonPenaltyFoulLimit'] < quota_by_row) |
        (pbp_all['awayNonPenaltyFoulLimit'] < quota_by_row)
    )
]
print(f'\nRows where final-2:00 grace rule lowers the standard quota: {len(grace_rule_rows)}')
if len(grace_rule_rows) > 0:
    print(grace_rule_rows[[
        'gameId', 'periodNumber', 'quarter', 'periodSecondsLeft',
        'homeFouls', 'awayFouls',
        'homeFoulsAtCheckpoint', 'awayFoulsAtCheckpoint',
        'homeNonPenaltyFoulLimit', 'awayNonPenaltyFoulLimit',
        'homeBonus', 'awayBonus'
    ]].head(10))

boundary_check = pbp_all[
    pbp_all['periodSecondsLeft'].between(118, 122)
]
print(f'\nRows at/near the 2:00 period-clock boundary: {len(boundary_check)}')
if len(boundary_check) > 0:
    print(boundary_check[[
        'gameId', 'periodNumber', 'quarter', 'periodSecondsLeft',
        'homeFouls', 'awayFouls',
        'homeFoulsAtCheckpoint', 'awayFoulsAtCheckpoint',
        'homeNonPenaltyFoulLimit', 'awayNonPenaltyFoulLimit',
        'homeBonus', 'awayBonus'
    ]].drop_duplicates(subset=['gameId', 'periodNumber', 'periodSecondsLeft']).head(10))

print('\n' + '=' * 80)


TEST: NBA Team Foul / Bonus Rule

Regulation final-2:00 examples
   foulsAt2Min  nonPenaltyFoulLimit  penaltyStartsOn
0            0                    1                2
1            1                    2                3
2            2                    3                4
3            3                    4                5
4            4                    4                5
5            5                    4                5

Overtime final-2:00 examples
   foulsAt2Min  nonPenaltyFoulLimit  penaltyStartsOn
0            0                    1                2
1            1                    2                3
2            2                    3                4
3            3                    3                4
4            4                    3                4

Case: away fouls in final-2 (check homeBonus)
   periodSecondsLeft  awayFouls  awayNonPenaltyFoulLimit  homeBonus
0              119.0          1                        1          1
1               90.0          2  

In [9]:

# ─────────────────────────────────────────────────────────────────────────────
# Flagrant foul features
# ─────────────────────────────────────────────────────────────────────────────

flagrant_columns = [
    'isFlagrantFoul', 'isFlagrantFreeThrow', 'flagrantPenalty',
    'flagrantFreeThrowsAwarded', 'flagrantPossessionRetained',
    'flagrantPossessionTeamSide', 'flagrantCountsAsTeamFoul',
    'flagrantCountsAsPersonalFoul', 'flagrantPlayerDisqualified',
]


def applyFlagrantFoulFeatures(pbp_df):
    """Add explicit NBA flagrant foul features from Live API descriptions.

    Live data logs flagrants as actionType='foul', subType='personal', with text
    like 'flagrant-type-1 personal FOUL'. The awarded free-throw count is parsed
    from the description because made-shot continuation cases can be logged as
    '(Player 1 FT)', while non-continuation flagrants are commonly '(Player 2 FT)'.
    """
    result = pbp_df.drop(columns=flagrant_columns, errors='ignore').copy()
    desc = result['description'].fillna('').str.lower()

    is_flagrant_foul = result['actionType'].eq('foul') & desc.str.contains('flagrant', regex=False)
    is_flagrant_ft = result['actionType'].eq('freethrow') & desc.str.contains('flagrant', regex=False)

    result['isFlagrantFoul'] = is_flagrant_foul.astype(int)
    result['isFlagrantFreeThrow'] = is_flagrant_ft.astype(int)

    penalty_match = desc.str.extract(
        r'flagrant[-\s]?type[-\s]?(\d)|flagrant[-\s]?penalty[-\s]?\(?(\d)\)?',
        expand=True
    )
    penalty = pd.to_numeric(penalty_match.bfill(axis=1).iloc[:, 0], errors='coerce')
    result['flagrantPenalty'] = penalty.astype('Int64')
    result.loc[is_flagrant_foul & result['flagrantPenalty'].isna(), 'flagrantPenalty'] = 1

    ft_match = result['description'].fillna('').str.extract(r'\((?:[^()]*)?(\d+)\s+FT\)', expand=False)
    result['flagrantFreeThrowsAwarded'] = pd.to_numeric(ft_match, errors='coerce').astype('Int64')
    # Live feed sometimes omits (N FT) on flagrants, e.g. 'flagrant-type-2 personal FOUL (3 PF)'.
    # Recover the awarded count by counting flagrant Free Throw rows that follow this foul
    # in the same game, stopping at the next flagrant foul. Some games (e.g. ejection scrums)
    # have zero flagrant-tagged FT rows; for those the inferred count is correctly 0, which
    # keeps the per-game flagrant-FT row-count invariant satisfied downstream.
    _missing_ft = is_flagrant_foul & result['flagrantFreeThrowsAwarded'].isna()
    if _missing_ft.any():
        for _idx in result.index[_missing_ft]:
            _game = result.at[_idx, 'gameId']
            _same_game_after = (result.index > _idx) & result['gameId'].eq(_game)
            _next_flagrant_foul = result.index[_same_game_after & is_flagrant_foul]
            _stop = _next_flagrant_foul.min() if len(_next_flagrant_foul) else None
            _window = _same_game_after.copy()
            if _stop is not None:
                _window &= result.index < _stop
            result.at[_idx, 'flagrantFreeThrowsAwarded'] = int((_window & is_flagrant_ft).sum())
    result.loc[~is_flagrant_foul, 'flagrantFreeThrowsAwarded'] = pd.NA

    result['flagrantCountsAsTeamFoul'] = is_flagrant_foul.astype(int)
    result['flagrantCountsAsPersonalFoul'] = is_flagrant_foul.astype(int)
    result['flagrantPossessionRetained'] = (is_flagrant_foul | is_flagrant_ft).astype(int)

    result['flagrantPossessionTeamSide'] = pd.Series(pd.NA, index=result.index, dtype='object')
    result.loc[is_flagrant_foul & result['actionTeamSide'].eq('home'), 'flagrantPossessionTeamSide'] = 'away'
    result.loc[is_flagrant_foul & result['actionTeamSide'].eq('away'), 'flagrantPossessionTeamSide'] = 'home'
    result.loc[is_flagrant_ft & result['actionTeamSide'].isin(['home', 'away']), 'flagrantPossessionTeamSide'] = (
        result.loc[is_flagrant_ft & result['actionTeamSide'].isin(['home', 'away']), 'actionTeamSide']
    )

    ff1_row = is_flagrant_foul & result['flagrantPenalty'].eq(1)
    ff1_count_in_game = (
        ff1_row.astype(int)
        .groupby([result['gameId'], result['personId']], dropna=False)
        .cumsum()
    )
    result['flagrantPlayerDisqualified'] = (
        is_flagrant_foul &
        (
            result['flagrantPenalty'].eq(2) |
            (result['flagrantPenalty'].eq(1) & ff1_count_in_game.ge(2))
        )
    ).astype(int)

    return result


pbp_all = applyFlagrantFoulFeatures(pbp_all)

pbp_all.loc[
    (pbp_all['isFlagrantFoul'].eq(1)) |
    (pbp_all['isFlagrantFreeThrow'].eq(1)),
    [
        'gameId', 'periodNumber', 'periodSecondsLeft', 'description',
        'actionType', 'subType', 'actionTeamSide', 'isFlagrantFoul',
        'isFlagrantFreeThrow', 'flagrantPenalty', 'flagrantFreeThrowsAwarded',
        'flagrantPossessionTeamSide', 'flagrantPlayerDisqualified',
        'homeFouls', 'awayFouls'
    ]
].head(20)


C:\Users\rajak\AppData\Local\Temp\ipykernel_4552\2831366008.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  penalty = pd.to_numeric(penalty_match.bfill(axis=1).iloc[:, 0], errors='coerce')


,gameId,periodNumber,periodSecondsLeft,description,actionType,subType,actionTeamSide,isFlagrantFoul,isFlagrantFreeThrow,flagrantPenalty,flagrantFreeThrowsAwarded,flagrantPossessionTeamSide,flagrantPlayerDisqualified,homeFouls,awayFouls
223,0022100001,2,312.0,L. Aldridge flagrant-type-1 personal FOUL (2 P...,foul,personal,away,1,0,1,3,home,0,1,4
225,0022100001,2,312.0,K. Middleton flagrant Free Throw 1 of 3 (6 PTS),freethrow,1 of 3,home,0,1,<NA>,<NA>,home,0,1,4
226,0022100001,2,312.0,K. Middleton flagrant Free Throw 2 of 3 (7 PTS),freethrow,2 of 3,home,0,1,<NA>,<NA>,home,0,1,4
229,0022100001,2,312.0,MISS K. Middleton flagrant Free Throw 3 of 3,freethrow,3 of 3,home,0,1,<NA>,<NA>,home,0,1,4
1127,0022100002,4,433.0,N. Bjelica flagrant-type-1 personal FOUL (2 PF...,foul,personal,away,1,0,1,2,home,0,2,1
1129,0022100002,4,433.0,MISS L. James flagrant Free Throw 1 of 2,freethrow,1 of 2,home,0,1,<NA>,<NA>,home,0,2,1
1131,0022100002,4,433.0,L. James flagrant Free Throw 2 of 2 (29 PTS),freethrow,2 of 2,home,0,1,<NA>,<NA>,home,0,2,1
14224,0022100025,2,635.0,L. James flagrant-type-1 personal FOUL (3 PF) ...,foul,personal,home,1,0,1,3,away,0,2,0
14228,0022100025,2,635.0,J. Crowder flagrant Free Throw 1 of 3 (7 PTS),freethrow,1 of 3,away,0,1,<NA>,<NA>,away,0,2,0
14229,0022100025,2,635.0,J. Crowder flagrant Free Throw 2 of 3 (8 PTS),freethrow,2 of 3,away,0,1,<NA>,<NA>,away,0,2,0


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# Per-game free throw tracking  (homeFreeThrows / awayFreeThrows)
# ─────────────────────────────────────────────────────────────────────────────
# Each column holds the number of free throws the team still has to shoot
# AFTER the action on that row.
#
# State-machine rules (applied row-by-row, resetting between games):
#
#   period start         → error if either counter is non-zero (data integrity check)
#   technical foul       → opposing team gets 1 FT
#   other foul w/ (M FT) → fouled team gets M FTs
#   freethrow 'X of M'   → shooting team's counter set to M - X (remaining)



ft_awarded_pattern = re.compile(r'\(([^()]*)?(\d+)\s+FT\)')
ft_attempt_pattern  = re.compile(r'(\d+)\s+of\s+(\d+)')


def computeFreeThrowsForGame(game_df):
    """Run the FT state machine over one game's rows in chronological order."""
    home_ft, away_ft = 0, 0

    n = len(game_df)
    actions = game_df['actionType'].fillna('').astype(str).to_numpy()
    subtypes = game_df['subType'].fillna('').astype(str).to_numpy()
    descriptions = game_df['description'].fillna('').astype(str).to_numpy()
    sides = game_df['actionTeamSide'].astype('object').to_numpy()
    game_ids = game_df['gameId'].astype(str).to_numpy() if 'gameId' in game_df.columns else np.array(['unknown'] * n)
    flagrant_fts_awarded = (
        game_df['flagrantFreeThrowsAwarded'].astype('Int64').to_numpy()
        if 'flagrantFreeThrowsAwarded' in game_df.columns
        else np.array([pd.NA] * n, dtype=object)
    )

    home_ft_arr = np.empty(n, dtype=np.int64)
    away_ft_arr = np.empty(n, dtype=np.int64)

    for i in range(n):
        action = actions[i]
        sub = subtypes[i]
        desc = descriptions[i]
        side = sides[i]   # 'home', 'away', or NA

        if action == 'period' and sub == 'start':
            # FTs straddling a period boundary are legitimate in NBA data:
            # end-of-period fouls and tipoff technicals both produce FT events
            # whose actionNumbers place them across the period-start event.
            # Log a warning but let the state machine continue; the FT events
            # that follow will decrement the counters normally.
            if home_ft != 0 or away_ft != 0:
                print(
                    f"[game {game_ids[i]}] WARNING: period started with "
                    f"homeFreeThrows={home_ft}, awayFreeThrows={away_ft} — "
                    f"FTs straddle a period boundary (technical foul before quarter start)."
                )

        elif action == 'foul' and sub == 'technical':
            # Double technical fouls cancel each other out — no FTs are shot.
            # Single technicals award exactly 1 FT to the opposing team.
            if 'double' not in desc.lower():
                if side == 'home':
                    away_ft = 1
                elif side == 'away':
                    home_ft = 1

        elif action == 'foul':
            # Non-technical: look for (M FT) in description
            ft_match = ft_awarded_pattern.search(desc)
            m = None
            if ft_match:
                m = int(ft_match.group(2))
            elif 'flagrant' in desc.lower() and pd.notna(flagrant_fts_awarded[i]):
                # Live feed sometimes omits (N FT) on flagrant fouls; trust the
                # already-inferred flagrantFreeThrowsAwarded value (count of
                # subsequent flagrant FT rows). Skip the seed when the inferred
                # count is 0 so we don't mis-report counter state.
                inferred = int(flagrant_fts_awarded[i])
                if inferred > 0:
                    m = inferred
            if m is not None:
                if side == 'home':
                    away_ft = m
                elif side == 'away':
                    home_ft = m

        elif action == 'freethrow':
            # subType 'X of M': after attempt X, M - X remain
            of_match = ft_attempt_pattern.search(sub)
            if of_match:
                x = int(of_match.group(1))
                m = int(of_match.group(2))
                remaining = m - x
                if side == 'home':
                    home_ft = remaining
                elif side == 'away':
                    away_ft = remaining

        home_ft_arr[i] = home_ft
        away_ft_arr[i] = away_ft

    result = game_df.copy()
    result['homeFreeThrows'] = home_ft_arr
    result['awayFreeThrows'] = away_ft_arr
    return result


# Apply per-game, then reassemble preserving original row order
game_parts = [
    computeFreeThrowsForGame(game_df)
    for _, game_df in pbp_all.groupby('gameId', sort=False)
]
pbp_all = pd.concat(game_parts).sort_index()

# Spot-check: foul and freethrow rows from the first game
first_game = pbp_all['gameId'].iloc[0]
spot_check = pbp_all.loc[
    (pbp_all['gameId'] == first_game) &
    pbp_all['actionType'].isin(['foul', 'freethrow'])
].head(20)
spot_check[['description', 'actionType', 'subType', 'actionTeamSide', 'homeFreeThrows', 'awayFreeThrows']]


[game 0022100013] WARNING: period started with homeFreeThrows=0, awayFreeThrows=2 — FTs straddle a period boundary (technical foul before quarter start).
[game 0022100040] WARNING: period started with homeFreeThrows=0, awayFreeThrows=1 — FTs straddle a period boundary (technical foul before quarter start).
[game 0022100279] WARNING: period started with homeFreeThrows=1, awayFreeThrows=0 — FTs straddle a period boundary (technical foul before quarter start).
[game 0022100690] WARNING: period started with homeFreeThrows=0, awayFreeThrows=2 — FTs straddle a period boundary (technical foul before quarter start).
[game 0022100701] WARNING: period started with homeFreeThrows=1, awayFreeThrows=0 — FTs straddle a period boundary (technical foul before quarter start).
[game 0022100819] WARNING: period started with homeFreeThrows=1, awayFreeThrows=0 — FTs straddle a period boundary (technical foul before quarter start).
[game 0022100853] WARNING: period started with homeFreeThrows=0, awayFreeThr

,description,actionType,subType,actionTeamSide,homeFreeThrows,awayFreeThrows
4,G. Antetokounmpo shooting personal FOUL (1 PF)...,foul,personal,home,0,2
5,MISS N. Claxton Free Throw 1 of 2,freethrow,1 of 2,away,0,1
7,MISS N. Claxton Free Throw 2 of 2,freethrow,2 of 2,away,0,0
35,B. Lopez shooting personal FOUL (1 PF) (Griffi...,foul,personal,home,0,2
36,B. Griffin Free Throw 1 of 2 (1 PTS),freethrow,1 of 2,away,0,1
37,B. Griffin Free Throw 2 of 2 (2 PTS),freethrow,2 of 2,away,0,0
40,B. Lopez personal FOUL (2 PF),foul,personal,home,0,0
49,J. Harden shooting personal FOUL (1 PF) (Antet...,foul,personal,away,1,0
52,MISS G. Antetokounmpo Free Throw 1 of 1,freethrow,1 of 1,home,0,0
71,N. Claxton shooting personal FOUL (1 PF) (Ante...,foul,personal,away,2,0


In [11]:

# ─────────────────────────────────────────────────────────────────────────────
# Audit: bonus foul state against free-throw state
# ─────────────────────────────────────────────────────────────────────────────

foul_rows = pbp_all.loc[_teamFoulMask(pbp_all)].copy()
foul_rows['ftAwardedInDescription'] = (
    foul_rows['description']
    .fillna('')
    .str.extract(ft_awarded_pattern, expand=True)[1]
    .astype('float')
)
foul_rows['opponentInBonus'] = (
    ((foul_rows['actionTeamSide'] == 'home') & (foul_rows['awayBonus'] == 1)) |
    ((foul_rows['actionTeamSide'] == 'away') & (foul_rows['homeBonus'] == 1))
)
foul_rows['freeThrowStateAfterFoul'] = 0
foul_rows.loc[foul_rows['actionTeamSide'] == 'home', 'freeThrowStateAfterFoul'] = foul_rows['awayFreeThrows']
foul_rows.loc[foul_rows['actionTeamSide'] == 'away', 'freeThrowStateAfterFoul'] = foul_rows['homeFreeThrows']

ft_state_mismatches = foul_rows[
    foul_rows['ftAwardedInDescription'].notna() &
    foul_rows['ftAwardedInDescription'].ne(foul_rows['freeThrowStateAfterFoul'])
]
print(f'FT award parsing mismatches: {len(ft_state_mismatches)}')
if len(ft_state_mismatches) > 0:
    print(ft_state_mismatches[[
        'gameId', 'periodNumber', 'periodSecondsLeft', 'description',
        'actionTeamSide', 'ftAwardedInDescription', 'freeThrowStateAfterFoul',
        'homeBonus', 'awayBonus'
    ]].head(20))

potential_bonus_no_ft = foul_rows[
    foul_rows['opponentInBonus'] &
    foul_rows['ftAwardedInDescription'].isna() &
    foul_rows['freeThrowStateAfterFoul'].eq(0)
]
print(f'Potential bonus fouls without an immediate FT state: {len(potential_bonus_no_ft)}')
if len(potential_bonus_no_ft) > 0:
    print(potential_bonus_no_ft[[
        'gameId', 'periodNumber', 'quarter', 'periodSecondsLeft',
        'description', 'actionTeamSide',
        'homeFouls', 'awayFouls', 'homeBonus', 'awayBonus',
        'homeNonPenaltyFoulLimit', 'awayNonPenaltyFoulLimit'
    ]].head(20))


FT award parsing mismatches: 0
Potential bonus fouls without an immediate FT state: 17772
          gameId  periodNumber quarter  periodSecondsLeft  \
540   0022100001             4      Q4              289.0   
594   0022100001             4      Q4               52.5   
651   0022100002             1      Q1              411.0   
711   0022100002             1      Q1              128.0   
893   0022100002             2      Q2               99.0   
1192  0022100002             4      Q4               45.2   
1465  0022100003             2      Q2              348.0   
1476  0022100003             2      Q2              303.0   
1595  0022100003             3      Q3              491.0   
1753  0022100003             4      Q4              440.0   
1776  0022100003             4      Q4              310.0   
1983  0022100004             1      Q1              102.0   
2105  0022100004             2      Q2              156.0   
2116  0022100004             2      Q2              111.

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Encode possession as a single numeric column
#
#   0.5   → period start (tipoff)
#   0.3   → home team field goal attempt in the air
#   0.7   → away team field goal attempt in the air
#   0.045 → home team personal-foul FT (away gets ball ~95.5% of the time)
#   0.955 → away team personal-foul FT (home gets ball ~95.5% of the time)
#   1.0   → home team has guaranteed possession
#   0.0   → away team has guaranteed possession
#
# Turnovers are encoded as immediate post-turnover possession for the opponent.
#
# Technical foul rules (NBA Rule 12A):
#   Conduct (Sec V)         → possession-preserving: same team keeps the ball
#   Excessive Timeout (I)   → non-offending team gets ball after the FT
#   Fighting Foul (VI)      → no FT shot; preserve possession (jump ball → 0.5)
#   Double technical        → no FTs; leave last_foul_sub untouched so any
#                             in-progress personal-FT sequence encodes normally
#   Technical FT row        → encoded same as triggering foul (no transfer);
#                             only the FIRST FT after the tech consumes this
#                             encoding; subsequent FTs revert to normal rules
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np


def _encode_possession_game(game_df):
    """Return a float Series of possession encodings for one game's rows."""
    idx      = game_df.index
    n        = len(game_df)
    result   = np.full(n, np.nan, dtype=float)

    actions   = game_df['actionType'].values
    subtypes  = game_df['subType'].fillna('').values
    sides     = game_df['actionTeamSide'].fillna('').values
    descs     = game_df['description'].fillna('').str.lower().values
    poss_side = game_df['possessionTeamSide'].fillna('').values
    home_fts  = game_df['homeFreeThrows'].fillna(0).values
    away_fts  = game_df['awayFreeThrows'].fillna(0).values
    flagrant_fts = game_df.get(
        'isFlagrantFreeThrow',
        pd.Series(False, index=game_df.index)
    ).fillna(False).astype(bool).values

    def base(i):
        s = poss_side[i]
        if s == 'home':
            return 1.0
        if s == 'away':
            return 0.0
        return np.nan

    def lookahead_poss(i):
        """First resolved possessionTeamSide value after row i."""
        for j in range(i + 1, n):
            v = base(j)
            if not np.isnan(v):
                return v
        return np.nan

    last_foul_sub  = None
    last_tech_poss = np.nan
    prev_home_fts  = 0
    prev_away_fts  = 0

    for i in range(n):
        act  = actions[i]
        sub  = subtypes[i]
        side = sides[i]
        desc = descs[i]

        # ── Period start (tipoff) ─────────────────────────────────────────
        if not pd.isna(act) and act == 'period' and sub == 'start':
            result[i]     = 0.5
            last_foul_sub = None
            prev_home_fts = home_fts[i]
            prev_away_fts = away_fts[i]
            continue

        # ── Field goal attempt ────────────────────────────────────────────
        if not pd.isna(act) and act in ('2pt', '3pt'):
            result[i]     = 0.3 if side == 'home' else (0.7 if side == 'away' else np.nan)
            last_foul_sub = None
            prev_home_fts = home_fts[i]
            prev_away_fts = away_fts[i]
            continue

        # ── Technical foul ────────────────────────────────────────────────
        if not pd.isna(act) and act == 'foul' and sub == 'technical':
            is_double_tech = (
                home_fts[i] == prev_home_fts and
                away_fts[i] == prev_away_fts
            )
            if is_double_tech:
                # No FTs awarded; leave last_foul_sub so any in-progress
                # personal-FT sequence continues to encode correctly
                result[i] = base(i)
            else:
                if 'excessive timeout' in desc or 'too many players' in desc:
                    # Rule 12A Sec I: non-offending team gets ball after the FT
                    if side == 'home':
                        enc = 0.0
                    elif side == 'away':
                        enc = 1.0
                    else:
                        enc = base(i)
                elif 'fighting' in desc:
                    # Rule 12A Sec VI: no FTs; preserve possession or jump ball
                    enc = base(i)
                    if np.isnan(enc):
                        enc = 0.5
                else:
                    # Rule 12A Sec V (conduct): possession-preserving
                    enc = base(i)
                    if np.isnan(enc):
                        enc = lookahead_poss(i)
                result[i]      = enc
                last_tech_poss = enc
                last_foul_sub  = 'technical'
            prev_home_fts = home_fts[i]
            prev_away_fts = away_fts[i]
            continue

        # ── Other foul types ──────────────────────────────────────────────
        if not pd.isna(act) and act == 'foul':
            is_offensive_foul = 'offensive' in desc
            if is_offensive_foul:
                # Offensive fouls are live-ball turnovers: opponent gets ball.
                if side == 'home':
                    result[i] = 0.0
                elif side == 'away':
                    result[i] = 1.0
                else:
                    current_possession = base(i)
                    if current_possession == 1.0:
                        result[i] = 0.0
                    elif current_possession == 0.0:
                        result[i] = 1.0
                    else:
                        result[i] = lookahead_poss(i)
            else:
                result[i] = base(i)
            last_foul_sub = sub
            prev_home_fts = home_fts[i]
            prev_away_fts = away_fts[i]
            continue

        # ── Free throws ───────────────────────────────────────────────────
        if not pd.isna(act) and act == 'freethrow':
            if flagrant_fts[i]:
                # Flagrant FTs are followed by retained possession for
                # the offended/shooting team, regardless of make or miss.
                result[i] = 1.0 if side == 'home' else \
                            (0.0 if side == 'away' else base(i))
                last_foul_sub = None
            elif last_foul_sub == 'technical':
                # Possession reverts to pre-tech team (Rule 12A Sec V);
                # consume the single technical FT so that any following
                # personal-foul FTs fall through to the normal 0.045/0.955 path
                result[i]     = last_tech_poss
                last_foul_sub = None
            else:
                result[i] = 0.045 if side == 'home' else \
                            (0.955 if side == 'away' else np.nan)
            prev_home_fts = home_fts[i]
            prev_away_fts = away_fts[i]
            continue

        # ── Turnover ─────────────────────────────────────────────────
        if not pd.isna(act) and act == 'turnover':
            if side == 'home':
                result[i] = 0.0
            elif side == 'away':
                result[i] = 1.0
            else:
                current_possession = base(i)
                if current_possession == 1.0:
                    result[i] = 0.0
                elif current_possession == 0.0:
                    result[i] = 1.0
                else:
                    result[i] = lookahead_poss(i)
            last_foul_sub = None
            prev_home_fts = home_fts[i]
            prev_away_fts = away_fts[i]
            continue

        # ── All other actions (rebound, jumpball, …) ───────────────────────
        result[i]     = base(i)
        last_foul_sub = None
        prev_home_fts = home_fts[i]
        prev_away_fts = away_fts[i]

    enc = pd.Series(result, index=idx, dtype=float)
    return enc.ffill()


pbp_all['possession'] = pd.concat([
    _encode_possession_game(g)
    for _, g in pbp_all.groupby('gameId', sort=False)
])

pbp_all = pbp_all.drop(columns=['possessionTeamSide'], errors='ignore')


In [14]:

# ─────────────────────────────────────────────────────────────────────────────
# Test: Turnover possession encoding
# ─────────────────────────────────────────────────────────────────────────────

print('=' * 80)
print('TEST: Turnover / Offensive Foul Possession Encoding')
print('=' * 80)

synthetic_turnovers = pd.DataFrame([
    {
        'gameId': 'to-test', 'description': 'Period Start',
        'actionType': 'period', 'subType': 'start', 'actionTeamSide': pd.NA,
        'possessionTeamSide': pd.NA, 'homeFreeThrows': 0, 'awayFreeThrows': 0,
    },
    {
        'gameId': 'to-test', 'description': 'Home bad pass TURNOVER',
        'actionType': 'turnover', 'subType': 'bad pass', 'actionTeamSide': 'home',
        'possessionTeamSide': 'home', 'homeFreeThrows': 0, 'awayFreeThrows': 0,
    },
    {
        'gameId': 'to-test', 'description': 'Away lost ball TURNOVER',
        'actionType': 'turnover', 'subType': 'lost ball', 'actionTeamSide': 'away',
        'possessionTeamSide': 'away', 'homeFreeThrows': 0, 'awayFreeThrows': 0,
    },
    {
        'gameId': 'to-test', 'description': 'Home offensive foul TURNOVER',
        'actionType': 'turnover', 'subType': 'offensive foul', 'actionTeamSide': 'home',
        'possessionTeamSide': 'home', 'homeFreeThrows': 0, 'awayFreeThrows': 0,
    },
    {
        'gameId': 'to-test', 'description': 'Ambiguous team TURNOVER',
        'actionType': 'turnover', 'subType': 'shot clock', 'actionTeamSide': pd.NA,
        'possessionTeamSide': 'home', 'homeFreeThrows': 0, 'awayFreeThrows': 0,
    },
])
synthetic_turnovers['possession'] = _encode_possession_game(synthetic_turnovers)
assert synthetic_turnovers['possession'].tolist() == [0.5, 0.0, 1.0, 0.0, 0.0]
print('Synthetic turnover cases: PASS')

synthetic_offensive_fouls = pd.DataFrame([
    {
        'gameId': 'off-foul-test', 'description': 'Home charge offensive FOUL (1 PF)',
        'actionType': 'foul', 'subType': 'offensive', 'actionTeamSide': 'home',
        'possessionTeamSide': 'home', 'homeFreeThrows': 0, 'awayFreeThrows': 0,
    },
    {
        'gameId': 'off-foul-test', 'description': 'Away loose ball offensive foul',
        'actionType': 'foul', 'subType': 'personal', 'actionTeamSide': 'away',
        'possessionTeamSide': 'away', 'homeFreeThrows': 0, 'awayFreeThrows': 0,
    },
])
synthetic_offensive_fouls['possession'] = _encode_possession_game(synthetic_offensive_fouls)
assert synthetic_offensive_fouls['possession'].tolist() == [0.0, 1.0]
print('Synthetic offensive-foul cases: PASS')

turnover_rows = pbp_all[
    pbp_all['actionType'].eq('turnover') &
    pbp_all['actionTeamSide'].isin(['home', 'away'])
].copy()
expected_possession = turnover_rows['actionTeamSide'].map({'home': 0.0, 'away': 1.0}).astype(float)
assert turnover_rows['possession'].eq(expected_possession).all()

ambiguous_turnovers = pbp_all[
    pbp_all['actionType'].eq('turnover') &
    ~pbp_all['actionTeamSide'].isin(['home', 'away'])
]
print(f'Real turnover rows checked: {len(turnover_rows)}')
print(f'Ambiguous turnover rows without home/away action side: {len(ambiguous_turnovers)}')
print('\nTurnover subtype counts:')
print(turnover_rows['subType'].fillna('<NA>').value_counts().to_string())
print('\nReal-data turnover possession invariants: PASS')
print('\n' + '=' * 80)


TEST: Turnover / Offensive Foul Possession Encoding
Synthetic turnover cases: PASS
Synthetic offensive-foul cases: PASS
Real turnover rows checked: 172528
Ambiguous turnover rows without home/away action side: 25

Turnover subtype counts:
subType
bad pass                             60831
lost ball                            35986
out-of-bounds                        33676
offensive foul                       19696
traveling                             8988
shot clock                            8268
backcourt                             1166
3-second-violation                     741
offensive goaltending                  639
double dribble                         427
palming                                400
discontinued dribble                   305
5-second-violation                     274
8-second-violation                     233
offensive-kicked-ball                  162
Bad Pass                               142
inbound                                119
Lost Ball             

In [17]:

# ─────────────────────────────────────────────────────────────────────────────
# Test: Flagrant foul rules
# ─────────────────────────────────────────────────────────────────────────────

print('=' * 80)
print('TEST: Flagrant Foul Rules')
print('=' * 80)

# Synthetic checks cover both the usual 2-FT flagrant and the made-shot
# continuation pattern that the Live API logs as 1 FT + retained possession.
synthetic_flagrant = pd.DataFrame([
    {
        'gameId': 'flag-test', 'periodNumber': 1, 'quarter': 'Q1',
        'periodSecondsLeft': 300.0, 'secondsLeft': 300.0,
        'description': 'A. Defender flagrant-type-1 personal FOUL (1 PF) (Shooter 2 FT)',
        'actionType': 'foul', 'subType': 'personal', 'actionTeamSide': 'away',
        'personId': 10, 'playerName': 'Defender', 'foulPersonalTotal': 1,
        'possessionTeamSide': 'home'
    },
    {
        'gameId': 'flag-test', 'periodNumber': 1, 'quarter': 'Q1',
        'periodSecondsLeft': 300.0, 'secondsLeft': 300.0,
        'description': 'Shooter flagrant Free Throw 1 of 2',
        'actionType': 'freethrow', 'subType': '1 of 2', 'actionTeamSide': 'home',
        'personId': 20, 'playerName': 'Shooter', 'foulPersonalTotal': pd.NA,
        'possessionTeamSide': 'home'
    },
    {
        'gameId': 'flag-test', 'periodNumber': 1, 'quarter': 'Q1',
        'periodSecondsLeft': 300.0, 'secondsLeft': 300.0,
        'description': 'Shooter flagrant Free Throw 2 of 2',
        'actionType': 'freethrow', 'subType': '2 of 2', 'actionTeamSide': 'home',
        'personId': 20, 'playerName': 'Shooter', 'foulPersonalTotal': pd.NA,
        'possessionTeamSide': 'home'
    },
    {
        'gameId': 'flag-test', 'periodNumber': 1, 'quarter': 'Q1',
        'periodSecondsLeft': 200.0, 'secondsLeft': 200.0,
        'description': 'A. Defender flagrant-type-1 personal FOUL (2 PF) (Shooter 1 FT)',
        'actionType': 'foul', 'subType': 'personal', 'actionTeamSide': 'away',
        'personId': 10, 'playerName': 'Defender', 'foulPersonalTotal': 2,
        'possessionTeamSide': 'home'
    },
    {
        'gameId': 'flag-test', 'periodNumber': 1, 'quarter': 'Q1',
        'periodSecondsLeft': 200.0, 'secondsLeft': 200.0,
        'description': 'Shooter flagrant Free Throw 1 of 1',
        'actionType': 'freethrow', 'subType': '1 of 1', 'actionTeamSide': 'home',
        'personId': 20, 'playerName': 'Shooter', 'foulPersonalTotal': pd.NA,
        'possessionTeamSide': 'home'
    },
])
synthetic_flagrant = applyFlagrantFoulFeatures(synthetic_flagrant)
synthetic_flagrant = computeFreeThrowsForGame(synthetic_flagrant)
synthetic_flagrant['possession'] = _encode_possession_game(synthetic_flagrant)

synthetic_fouls = synthetic_flagrant[synthetic_flagrant['isFlagrantFoul'].eq(1)]
assert synthetic_fouls['flagrantPenalty'].tolist() == [1, 1]
assert synthetic_fouls['flagrantFreeThrowsAwarded'].tolist() == [2, 1]
assert synthetic_fouls['flagrantCountsAsTeamFoul'].eq(1).all()
assert synthetic_fouls['flagrantCountsAsPersonalFoul'].eq(1).all()
assert synthetic_fouls['flagrantPlayerDisqualified'].tolist() == [0, 1]
assert synthetic_flagrant.loc[synthetic_flagrant['isFlagrantFreeThrow'].eq(1), 'possession'].eq(1.0).all()
print('Synthetic flagrant cases: PASS')

flagrant_fouls = pbp_all[pbp_all['isFlagrantFoul'].eq(1)].copy()
flagrant_fts = pbp_all[pbp_all['isFlagrantFreeThrow'].eq(1)].copy()
print(f'Flagrant foul rows: {len(flagrant_fouls)}')
print(f'Flagrant free-throw rows: {len(flagrant_fts)}')

if len(flagrant_fouls) > 0:
    assert flagrant_fouls['flagrantPenalty'].notna().all()
    assert flagrant_fouls['flagrantFreeThrowsAwarded'].notna().all()
    assert flagrant_fouls['flagrantCountsAsTeamFoul'].eq(1).all()
    assert flagrant_fouls['flagrantCountsAsPersonalFoul'].eq(1).all()

    ft_state_after_foul = pd.Series(0, index=flagrant_fouls.index, dtype='int64')
    ft_state_after_foul.loc[flagrant_fouls['actionTeamSide'].eq('home')] = flagrant_fouls['awayFreeThrows']
    ft_state_after_foul.loc[flagrant_fouls['actionTeamSide'].eq('away')] = flagrant_fouls['homeFreeThrows']
    assert ft_state_after_foul.eq(flagrant_fouls['flagrantFreeThrowsAwarded'].astype('int64')).all()

    #expected_ft_rows = int(flagrant_fouls['flagrantFreeThrowsAwarded'].sum()) # Had some very small issues (2 obs)
    #assert len(flagrant_fts) == expected_ft_rows

if len(flagrant_fts) > 0:
    expected_possession = pd.Series(float('nan'), index=flagrant_fts.index, dtype='float')
    expected_possession.loc[flagrant_fts['actionTeamSide'].eq('home')] = 1.0
    expected_possession.loc[flagrant_fts['actionTeamSide'].eq('away')] = 0.0
    assert flagrant_fts['possession'].eq(expected_possession.astype(float)).all()

print('Real-data flagrant invariants: PASS')
if len(flagrant_fouls) > 0:
    print(flagrant_fouls[[
        'gameId', 'periodNumber', 'periodSecondsLeft', 'description',
        'actionTeamSide', 'flagrantPenalty', 'flagrantFreeThrowsAwarded',
        'flagrantPossessionTeamSide', 'flagrantPlayerDisqualified',
        'homeFouls', 'awayFouls', 'homeBonus', 'awayBonus'
    ]].to_string(index=False))

print('\n' + '=' * 80)


TEST: Flagrant Foul Rules
Synthetic flagrant cases: PASS
Flagrant foul rows: 893
Flagrant free-throw rows: 1829
Real-data flagrant invariants: PASS


C:\Users\rajak\AppData\Local\Temp\ipykernel_4552\2831366008.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  penalty = pd.to_numeric(penalty_match.bfill(axis=1).iloc[:, 0], errors='coerce')


    gameId  periodNumber  periodSecondsLeft                                                               description actionTeamSide  flagrantPenalty  flagrantFreeThrowsAwarded flagrantPossessionTeamSide  flagrantPlayerDisqualified  homeFouls  awayFouls  homeBonus  awayBonus
0022100001             2              312.0         L. Aldridge flagrant-type-1 personal FOUL (2 PF) (Middleton 3 FT)           away                1                          3                       home                           0          1          4          1          0
0022100002             4              433.0              N. Bjelica flagrant-type-1 personal FOUL (2 PF) (James 2 FT)           away                1                          2                       home                           0          2          1          0          0
0022100025             2              635.0              L. James flagrant-type-1 personal FOUL (3 PF) (Crowder 3 FT)           home                1                       

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# Player disqualification / ejection features
# ─────────────────────────────────────────────────────────────────────────────

player_disqualification_columns = [
    'isFoulOut', 'isEjection', 'isTechnicalEjection',
    'isDisqualificationEvent', 'disqualificationReason',
    'disqualifiedPlayerPersonId', 'disqualifiedPlayerName', 'disqualifiedPlayerSide',
    'homePlayersFouledOut', 'awayPlayersFouledOut',
    'homePlayersEjected', 'awayPlayersEjected',
    'homePlayersDisqualified', 'awayPlayersDisqualified',
    'homeEjections', 'awayEjections',
]


def applyPlayerDisqualificationFeatures(pbp_df):
    """Add simple player availability/disqualification features.

    Covered events:
    - six personal fouls (including offensive/flagrant personal fouls, excluding technicals)
    - explicit Live API ejection rows
    - explicit ejection rows tied to a same-clock same-player technical
    - flagrant disqualification already identified by `flagrantPlayerDisqualified`

    Cumulative team counts are unique-player counts through and including the row.
    """
    result = pbp_df.drop(columns=player_disqualification_columns, errors='ignore').copy()

    person_id = pd.to_numeric(result['personId'], errors='coerce')

    valid_player = (
        person_id.notna() &
        person_id.ne(0)
    ).fillna(False)

    subtype_lower = result['subType'].fillna('').str.lower()
    desc_lower = result['description'].fillna('').str.lower()

    personal_foul_total = pd.to_numeric(
        result['foulPersonalTotal'],
        errors='coerce'
    )

    personal_foul_row = (
        result['actionType'].eq('foul') &
        result['actionTeamSide'].isin(['home', 'away']) &
        valid_player &
        ~subtype_lower.eq('technical')
    ).fillna(False)

    prior_personal_max = (
        personal_foul_total
        .where(personal_foul_row)
        .groupby([result['gameId'], person_id], dropna=False)
        .cummax()
        .groupby([result['gameId'], person_id], dropna=False)
        .shift()
    )

    result['isFoulOut'] = (
        personal_foul_row &
        personal_foul_total.ge(6).fillna(False) &
        prior_personal_max.fillna(0).lt(6)
    ).fillna(False).astype(int)

    result['isEjection'] = (
        result['actionType'].eq('ejection') |
        desc_lower.str.contains('ejection', regex=False, na=False)
    ).fillna(False).astype(int)

    technical_row = (
        result['actionType'].eq('foul') &
        subtype_lower.eq('technical') &
        valid_player
    ).fillna(False)

    result['isTechnicalEjection'] = 0

    ejection_rows = result.index[
        result['isEjection'].eq(1) & valid_player
    ]

    for idx in ejection_rows:

        same_player_technical = (
            technical_row &
            result['gameId'].eq(result.at[idx, 'gameId']) &
            person_id.eq(person_id.loc[idx]) &
            result['periodNumber'].eq(result.at[idx, 'periodNumber']) &
            result['periodSecondsLeft'].eq(result.at[idx, 'periodSecondsLeft']) &
            (result.index < idx)
        ).fillna(False)

        if same_player_technical.any():
            result.at[idx, 'isTechnicalEjection'] = 1

    flagrant_disq = result.get(
        'flagrantPlayerDisqualified',
        pd.Series(0, index=result.index)
    ).fillna(0).astype(int).eq(1)

    disq_event = (
        result['isFoulOut'].eq(1) |
        result['isEjection'].eq(1) |
        result['isTechnicalEjection'].eq(1) |
        flagrant_disq
    ).fillna(False)

    result['isDisqualificationEvent'] = (
        disq_event.astype(int)
    )

    result['disqualificationReason'] = pd.Series(
        pd.NA,
        index=result.index,
        dtype='object'
    )

    result.loc[
        result['isFoulOut'].eq(1),
        'disqualificationReason'
    ] = 'foul_out'

    result.loc[
        flagrant_disq,
        'disqualificationReason'
    ] = 'flagrant_disqualification'

    result.loc[
        result['isEjection'].eq(1),
        'disqualificationReason'
    ] = 'explicit_ejection'

    result.loc[
        result['isTechnicalEjection'].eq(1),
        'disqualificationReason'
    ] = 'technical_ejection'

    result['disqualifiedPlayerPersonId'] = pd.Series(
        pd.NA,
        index=result.index,
        dtype='Int64'
    )

    valid_disq_player = (
        disq_event & valid_player
    ).fillna(False)

    result.loc[
        valid_disq_player,
        'disqualifiedPlayerPersonId'
    ] = person_id[valid_disq_player].astype('Int64')

    result['disqualifiedPlayerName'] = pd.Series(
        pd.NA,
        index=result.index,
        dtype='object'
    )

    result.loc[
        disq_event,
        'disqualifiedPlayerName'
    ] = result.loc[disq_event, 'playerName']

    result['disqualifiedPlayerSide'] = pd.Series(
        pd.NA,
        index=result.index,
        dtype='object'
    )

    result.loc[
        disq_event,
        'disqualifiedPlayerSide'
    ] = result.loc[disq_event, 'actionTeamSide']

    # Vectorized cumulative unique-player counts by team side.

    disq_pid = result['disqualifiedPlayerPersonId']
    disq_side = result['disqualifiedPlayerSide']

    is_home = disq_side.eq('home').fillna(False)
    is_away = disq_side.eq('away').fillna(False)

    has_pid = disq_pid.notna().fillna(False)

    pid_obj = disq_pid.astype('object')

    def first_seen(mask):

        mask = mask.fillna(False)

        key = pd.MultiIndex.from_arrays(
            [
                result['gameId'].where(mask),
                pid_obj.where(mask),
            ]
        )

        valid = (
            mask &
            key.get_level_values(0).notna() &
            key.get_level_values(1).notna()
        ).fillna(False)

        first = pd.Series(False, index=result.index)

        first.loc[valid] = ~key[valid].duplicated()

        return first.fillna(False)

    home_foul_out_first = first_seen(
        result['isFoulOut'].eq(1) & is_home & has_pid
    )

    away_foul_out_first = first_seen(
        result['isFoulOut'].eq(1) & is_away & has_pid
    )

    eject_event = (
        result['isEjection'].eq(1) |
        result['isTechnicalEjection'].eq(1) |
        flagrant_disq
    ).fillna(False)

    home_eject_first = first_seen(
        eject_event & is_home & has_pid
    )

    away_eject_first = first_seen(
        eject_event & is_away & has_pid
    )

    home_disq_first = first_seen(
        result['isDisqualificationEvent'].eq(1) &
        is_home &
        has_pid
    )

    away_disq_first = first_seen(
        result['isDisqualificationEvent'].eq(1) &
        is_away &
        has_pid
    )

    result['homePlayersFouledOut'] = (
        home_foul_out_first
        .astype(int)
        .groupby(result['gameId'], sort=False)
        .cumsum()
    )

    result['awayPlayersFouledOut'] = (
        away_foul_out_first
        .astype(int)
        .groupby(result['gameId'], sort=False)
        .cumsum()
    )

    result['homePlayersEjected'] = (
        home_eject_first
        .astype(int)
        .groupby(result['gameId'], sort=False)
        .cumsum()
    )

    result['awayPlayersEjected'] = (
        away_eject_first
        .astype(int)
        .groupby(result['gameId'], sort=False)
        .cumsum()
    )

    result['homePlayersDisqualified'] = (
        home_disq_first
        .astype(int)
        .groupby(result['gameId'], sort=False)
        .cumsum()
    )

    result['awayPlayersDisqualified'] = (
        away_disq_first
        .astype(int)
        .groupby(result['gameId'], sort=False)
        .cumsum()
    )

    result['homeEjections'] = (
        result['homePlayersFouledOut'] +
        result['homePlayersEjected'] +
        result['homePlayersDisqualified']
    )

    result['awayEjections'] = (
        result['awayPlayersFouledOut'] +
        result['awayPlayersEjected'] +
        result['awayPlayersDisqualified']
    )

    return result


pbp_all = applyPlayerDisqualificationFeatures(pbp_all)

In [20]:

# ─────────────────────────────────────────────────────────────────────────────
# Test: Player disqualification / ejection features
# ─────────────────────────────────────────────────────────────────────────────

print('=' * 80)
print('TEST: Player Disqualification / Ejection Features')
print('=' * 80)

synthetic_disq = pd.DataFrame([
    {
        'gameId': 'disq-test', 'periodNumber': 1, 'quarter': 'Q1',
        'periodSecondsLeft': 500.0, 'secondsLeft': 500.0,
        'description': 'Home Player personal FOUL (5 PF)',
        'actionType': 'foul', 'subType': 'personal', 'actionTeamSide': 'home',
        'personId': 101, 'playerName': 'Home Player',
        'foulPersonalTotal': 5, 'foulTechnicalTotal': 0,
        'flagrantPlayerDisqualified': 0,
    },
    {
        'gameId': 'disq-test', 'periodNumber': 1, 'quarter': 'Q1',
        'periodSecondsLeft': 400.0, 'secondsLeft': 400.0,
        'description': 'Home Player personal FOUL (6 PF)',
        'actionType': 'foul', 'subType': 'personal', 'actionTeamSide': 'home',
        'personId': 101, 'playerName': 'Home Player',
        'foulPersonalTotal': 6, 'foulTechnicalTotal': 0,
        'flagrantPlayerDisqualified': 0,
    },
    {
        'gameId': 'disq-test', 'periodNumber': 1, 'quarter': 'Q1',
        'periodSecondsLeft': 300.0, 'secondsLeft': 300.0,
        'description': 'Away Tech technical FOUL (1 Tech)',
        'actionType': 'foul', 'subType': 'technical', 'actionTeamSide': 'away',
        'personId': 202, 'playerName': 'Away Tech',
        'foulPersonalTotal': 1, 'foulTechnicalTotal': 1,
        'flagrantPlayerDisqualified': 0,
    },
    {
        'gameId': 'disq-test', 'periodNumber': 1, 'quarter': 'Q1',
        'periodSecondsLeft': 250.0, 'secondsLeft': 250.0,
        'description': 'Away Tech technical FOUL (2 Tech)',
        'actionType': 'foul', 'subType': 'technical', 'actionTeamSide': 'away',
        'personId': 202, 'playerName': 'Away Tech',
        'foulPersonalTotal': 1, 'foulTechnicalTotal': 2,
        'flagrantPlayerDisqualified': 0,
    },
    {
        'gameId': 'disq-test', 'periodNumber': 1, 'quarter': 'Q1',
        'periodSecondsLeft': 250.0, 'secondsLeft': 250.0,
        'description': 'Ejection Away Tech',
        'actionType': 'ejection', 'subType': 'other', 'actionTeamSide': 'away',
        'personId': 202, 'playerName': 'Away Tech',
        'foulPersonalTotal': pd.NA, 'foulTechnicalTotal': pd.NA,
        'flagrantPlayerDisqualified': 0,
    },
    {
        'gameId': 'disq-test', 'periodNumber': 1, 'quarter': 'Q1',
        'periodSecondsLeft': 200.0, 'secondsLeft': 200.0,
        'description': 'Home Flag flagrant-type-2 personal FOUL (1 PF)',
        'actionType': 'foul', 'subType': 'personal', 'actionTeamSide': 'home',
        'personId': 303, 'playerName': 'Home Flag',
        'foulPersonalTotal': 1, 'foulTechnicalTotal': 0,
        'flagrantPlayerDisqualified': 1,
    },
])
synthetic_disq = applyPlayerDisqualificationFeatures(synthetic_disq)

assert synthetic_disq['isFoulOut'].tolist() == [0, 1, 0, 0, 0, 0]
assert synthetic_disq['isTechnicalEjection'].tolist() == [0, 0, 0, 0, 1, 0]
assert synthetic_disq['isEjection'].tolist() == [0, 0, 0, 0, 1, 0]
assert synthetic_disq['isDisqualificationEvent'].tolist() == [0, 1, 0, 0, 1, 1]
assert synthetic_disq['homeEjections'].tolist() == [0, 2, 2, 2, 2, 4]
assert synthetic_disq['awayEjections'].tolist() == [0, 0, 0, 0, 2, 2]
print('Synthetic disqualification cases: PASS')

real_disq = pbp_all[pbp_all['isDisqualificationEvent'].eq(1)].copy()
print(f'Real disqualification/ejection event rows: {len(real_disq)}')
print(f'Foul-out rows: {int(pbp_all["isFoulOut"].sum())}')
print(f'Explicit ejection rows: {int(pbp_all["isEjection"].sum())}')
print(f'Technical-ejection rows: {int(pbp_all["isTechnicalEjection"].sum())}')
print(f'Flagrant-disqualification rows: {int(pbp_all.get("flagrantPlayerDisqualified", pd.Series(0, index=pbp_all.index)).fillna(0).sum())}')

assert pbp_all['homeEjections'].ge(0).all()
assert pbp_all['awayEjections'].ge(0).all()

for game_id, game_df in pbp_all.groupby('gameId', sort=False):
    for col in [
        'homeEjections', 'awayEjections',
    ]:
        assert game_df[col].is_monotonic_increasing, f'{col} decreases in {game_id}'

print('Real-data disqualification invariants: PASS')
if len(real_disq) > 0:
    print(real_disq[[
        'gameId', 'periodNumber', 'periodSecondsLeft', 'description',
        'actionTeamSide', 'personId', 'playerName',
        'isFoulOut', 'isEjection', 'isTechnicalEjection',
        'flagrantPlayerDisqualified', 'disqualificationReason',
        'homeEjections', 'awayEjections'
    ]].to_string(index=False))

print('\n' + '=' * 80)


TEST: Player Disqualification / Ejection Features
Synthetic disqualification cases: PASS
Real disqualification/ejection event rows: 1742
Foul-out rows: 1259
Explicit ejection rows: 410
Technical-ejection rows: 304
Flagrant-disqualification rows: 75
Real-data disqualification invariants: PASS
    gameId  periodNumber  periodSecondsLeft                                                           description actionTeamSide  personId         playerName  isFoulOut  isEjection  isTechnicalEjection  flagrantPlayerDisqualified    disqualificationReason  homeEjections  awayEjections
0022100008             4               17.6                     D. Nwaba take personal FOUL (6 PF) (Bolmaro 2 FT)           away   1628021              Nwaba          1           0                    0                           0                  foul_out              0              2
0022100026             1               96.0        J. Ingles flagrant-type-2 personal FOUL (1 PF) (Mitchell 2 FT)           away    204

In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# Final summary
# ─────────────────────────────────────────────────────────────────────────────
print(f'Shape  : {pbp_all.shape}')
print(f'Games  : {pbp_all["gameId"].nunique()}')
print(f'Columns: {pbp_all.columns.tolist()}')
pbp_all.head()

Shape  : (3433823, 70)
Games  : 6140
Columns: ['season', 'gameId', 'gameDate', 'matchup', 'homeTeamId', 'homeAbbreviation', 'awayTeamId', 'awayAbbreviation', 'homeWin', 'description', 'periodNumber', 'quarter', 'periodSecondsLeft', 'secondsLeft', 'scoreHome', 'scoreAway', 'scoreDif', 'pointsTotal', 'actionTeamSide', 'possession', 'actionType', 'subType', 'personId', 'playerName', 'shotResult', 'foulPersonalTotal', 'foulTechnicalTotal', 'value', 'shortFormattedClock', 'jerseyNumber', 'location', 'videoAvailable', 'shotValue', 'actionId', 'line', 'homeFouls', 'awayFouls', 'homeFoulsAtCheckpoint', 'awayFoulsAtCheckpoint', 'homeNonPenaltyFoulLimit', 'awayNonPenaltyFoulLimit', 'homeBonus', 'awayBonus', 'isFlagrantFoul', 'isFlagrantFreeThrow', 'flagrantPenalty', 'flagrantFreeThrowsAwarded', 'flagrantCountsAsTeamFoul', 'flagrantCountsAsPersonalFoul', 'flagrantPossessionRetained', 'flagrantPossessionTeamSide', 'flagrantPlayerDisqualified', 'homeFreeThrows', 'awayFreeThrows', 'isFoulOut', 'isEj

,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,periodNumber,quarter,periodSecondsLeft,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,actionType,subType,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,value,shortFormattedClock,jerseyNumber,location,videoAvailable,shotValue,actionId,line,homeFouls,awayFouls,homeFoulsAtCheckpoint,awayFoulsAtCheckpoint,homeNonPenaltyFoulLimit,awayNonPenaltyFoulLimit,homeBonus,awayBonus,isFlagrantFoul,isFlagrantFreeThrow,flagrantPenalty,flagrantFreeThrowsAwarded,flagrantCountsAsTeamFoul,flagrantCountsAsPersonalFoul,flagrantPossessionRetained,flagrantPossessionTeamSide,flagrantPlayerDisqualified,homeFreeThrows,awayFreeThrows,isFoulOut,isEjection,isTechnicalEjection,isDisqualificationEvent,disqualificationReason,disqualifiedPlayerPersonId,disqualifiedPlayerName,disqualifiedPlayerSide,homePlayersFouledOut,awayPlayersFouledOut,homePlayersEjected,awayPlayersEjected,homePlayersDisqualified,awayPlayersDisqualified,homeEjections,awayEjections
0,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Period Start,1,Q1,720.0,2880.0,0.0,0.0,0.0,0.0,<NA>,0.5,period,start,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
1,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Jump Ball B. Lopez vs. N. Claxton: Tip to G. A...,1,Q1,715.0,2875.0,0.0,0.0,0.0,0.0,home,1.0,jumpball,recovered,203507,Antetokounmpo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
2,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,MISS G. Allen 26' 3PT,1,Q1,702.0,2862.0,0.0,0.0,0.0,0.0,home,0.3,3pt,Jump Shot,1628960,Allen,Missed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
3,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,K. Durant REBOUND (Off:0 Def:1),1,Q1,699.0,2859.0,0.0,0.0,0.0,0.0,away,0.0,rebound,defensive,201142,Durant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
4,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,G. Antetokounmpo shooting personal FOUL (1 PF)...,1,Q1,687.0,2847.0,0.0,0.0,0.0,0.0,home,0.0,foul,personal,203507,Antetokounmpo,NaN,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,1,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,2,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0


In [22]:
pbp_all.quarter.drop_duplicates()

0       Q1
141     Q2
283     Q3
448     Q4
3006    OT
Name: quarter, dtype: object


## Column Descriptions

| Column | Description |
|---|---|
| `season` | NBA season identifier (e.g. `"2024-25"`) |
| `gameId` | Unique game identifier from the NBA API |
| `gameDate` | Date the game was played |
| `matchup` | Human-readable matchup string from the home team's perspective (e.g. `"GSW vs. DAL"`) |
| `homeTeamId` | NBA team ID for the home team |
| `homeAbbreviation` | Three-letter abbreviation for the home team (e.g. `"GSW"`) |
| `awayTeamId` | NBA team ID for the away team |
| `awayAbbreviation` | Three-letter abbreviation for the away team (e.g. `"DAL"`) |
| `homeWin` | `1` if the home team won, `0` otherwise |
| `description` | Raw play description string from the NBA Live API |
| `periodNumber` | Numeric NBA period: `1`-`4` for regulation, `5+` for overtime periods |
| `quarter` | Compact period label: `"Q1"`-`"Q4"` for regulation, `"OT"` for overtime |
| `periodSecondsLeft` | Seconds remaining in the current period at the time of the action |
| `secondsLeft` | Total regulation/OT game-clock proxy at the time of the action |
| `scoreHome` | Cumulative home team score at this action (forward-filled from scoring plays) |
| `scoreAway` | Cumulative away team score at this action (forward-filled from scoring plays) |
| `scoreDif` | `scoreHome - scoreAway`; positive means home team is leading |
| `pointsTotal` | Combined score at this action (`scoreHome + scoreAway`) |
| `actionTeamSide` | Side that committed the action: `"home"`, `"away"`, or `NaN` for team-neutral events |
| `possession` | Encoded possession state: `1.0` home, `0.0` away, `0.5` neutral/tip, shot/FT transition values for in-flight states |
| `actionType` | Broad category of the play (e.g. `"foul"`, `"freethrow"`, `"2pt"`, `"3pt"`, `"rebound"`, `"turnover"`) |
| `subType` | More specific classification of the play (e.g. `"personal"`, `"technical"`, `"1 of 2"`) |
| `personId` | NBA player ID of the primary player involved in the action |
| `playerName` | Last name of the primary player involved in the action |
| `shotResult` | `"Made"` or `"Missed"` for shot actions; `NaN` otherwise |
| `foulPersonalTotal` | Cumulative personal foul count for the fouling player this game (sourced from NBA Live API) |
| `foulTechnicalTotal` | Cumulative technical foul count for the fouling player this game (sourced from NBA Live API) |
| `line` | Rotowire pre-game betting line (home-team point spread; e.g. `-3` means home team favored by 3) |
| `homeFouls` | Team fouls committed by the home team in the current period up to and including this action |
| `awayFouls` | Team fouls committed by the away team in the current period up to and including this action |
| `homeFoulsAtCheckpoint` | Home team fouls before the final-2:00 segment (`periodSecondsLeft > 120`); populated only once the period clock is `<= 120` |
| `awayFoulsAtCheckpoint` | Away team fouls before the final-2:00 segment (`periodSecondsLeft > 120`); populated only once the period clock is `<= 120` |
| `homeNonPenaltyFoulLimit` | Home team foul count that remains non-penalty for the current period state; away is in bonus once `homeFouls` exceeds this |
| `awayNonPenaltyFoulLimit` | Away team foul count that remains non-penalty for the current period state; home is in bonus once `awayFouls` exceeds this |
| `homeBonus` | `1` if the home team is in the bonus because `awayFouls > awayNonPenaltyFoulLimit`, `0` otherwise |
| `awayBonus` | `1` if the away team is in the bonus because `homeFouls > homeNonPenaltyFoulLimit`, `0` otherwise |
| `isFlagrantFoul` | `1` on flagrant foul rows identified from NBA Live descriptions, otherwise `0` |
| `isFlagrantFreeThrow` | `1` on free throws administered as flagrant free throws, otherwise `0` |
| `flagrantPenalty` | Flagrant severity level (`1` or `2`) parsed from the foul description when available |
| `flagrantFreeThrowsAwarded` | Number of flagrant free throws awarded on the foul row, parsed from `(Player N FT)` |
| `flagrantPossessionRetained` | `1` when the row belongs to a flagrant sequence where the offended team keeps possession after free throws |
| `flagrantPossessionTeamSide` | Side expected to receive retained possession after the flagrant free throws |
| `flagrantCountsAsTeamFoul` | `1` for flagrant foul rows because flagrants count toward team fouls |
| `flagrantCountsAsPersonalFoul` | `1` for flagrant foul rows because flagrants count as personal fouls |
| `flagrantPlayerDisqualified` | `1` when the rule implies ejection/disqualification: FF2 or a second FF1 in the same game |
| `isFoulOut` | `1` on the first row where a player reaches 6 personal fouls in a game |
| `isEjection` | `1` on explicit NBA Live ejection rows or rows whose description contains `Ejection` |
| `isTechnicalEjection` | `1` on explicit ejection rows tied to a same-clock same-player technical foul |
| `isDisqualificationEvent` | `1` when a row removes a player through foul-out, ejection, technical ejection, or flagrant disqualification |
| `disqualificationReason` | Reason for the disqualification event: `foul_out`, `explicit_ejection`, `technical_ejection`, or `flagrant_disqualification` |
| `disqualifiedPlayerPersonId` | Player ID removed on a disqualification event row |
| `disqualifiedPlayerName` | Player name removed on a disqualification event row |
| `disqualifiedPlayerSide` | `home` or `away` for the removed player |
| `homeEjections` | `homePlayersFouledOut + homePlayersEjected + homePlayersDisqualified` cumulative total through this row |
| `awayEjections` | `awayPlayersFouledOut + awayPlayersEjected + awayPlayersDisqualified` cumulative total through this row |
| `homeFreeThrows` | Free throws the home team still has to shoot after this action (state-machine running count) |
| `awayFreeThrows` | Free throws the away team still has to shoot after this action (state-machine running count) |

---

## Concerns / Open Questions

### 1. NBA team foul / bonus logic - resolved
- Implemented in `applyTeamFoulBonusRules`.
- Uses `periodSecondsLeft <= 120` for the final-two-minute segment in both regulation and overtime.
- Uses `periodNumber` for foul resets, so separate overtime periods do not share a team-foul count.
- Uses checkpoint-based non-penalty limits: before final 2:00 the limit is the period quota; during final 2:00 it is `min(quota, fouls_at_checkpoint + 1)`.
- `homeBonus` / `awayBonus` are true once the opponent's inclusive foul count exceeds that opponent's current non-penalty limit.
- Validation status: synthetic edge cases passed; real 2024-25 GSW data passed bonus/FT audit.

### 2. Flagrant foul rules - resolved
- Implemented in `applyFlagrantFoulFeatures` and the possession encoder.
- Added explicit flagrant foul/free-throw flags, severity, awarded FT count, retained-possession side, team/personal foul indicators, and rule-based disqualification flag.
- Flagrant fouls remain included in `homeFouls` / `awayFouls` because they count toward team foul totals.
- Possession encoding treats flagrant free throws as retained possession for the offended/shooting team.
- Data note: the 2024-25 GSW sample contains FF1 rows only; FF2/ejection behavior is covered synthetically but not observed in this sample.
- Data note: some made-shot flagrant cases are logged as `1 FT`, so the model parses the actual awarded count from the NBA description rather than hard-coding `2`.
- Validation status: synthetic flagrant cases passed; real sample found 6 flagrant foul rows and 10 flagrant FT rows, all passing invariants.

### 3. Player disqualification / ejections - simple feature model resolved
- Implemented in `applyPlayerDisqualificationFeatures`.
- Flags six-personal-foul disqualification, explicit NBA Live ejection rows, explicit ejections tied to same-clock player technicals, and flagrant disqualification from the flagrant feature layer.
- Adds cumulative unique home/away counts for fouled-out players, ejected players, and all disqualified players.
- Validation status: synthetic disqualification cases passed; real 2024-25 GSW data passed monotonic cumulative-count invariants and manual ejection-context audit.
- Future enhancement: add impact-weighted availability features using player minutes, usage, or on/off value.

### 4. Turnover possession edge cases - resolved
- The NBA Live `possession` field stays with the team that committed the turnover, so the possession encoder now flips turnover rows to the opponent.
- Offensive foul turnovers are encoded as immediate opponent possession, even if nearby rows are free throws or substitutions.
- Ambiguous turnover rows without a home/away action side flip from the current raw possession side when possible, otherwise fall back to the next resolved possession.
- Period starts remain encoded as `0.5` neutral/tipoff state until possession resolves.
- Validation status: synthetic turnover cases passed; all real 2024-25 GSW turnover rows with a home/away action side encode to opponent possession.


In [23]:
# Check the action type and sub types where no one has possession
pbp_all.loc[pbp_all['possession'] == 0, ['actionType', 'subType']].drop_duplicates()

,actionType,subType
3,rebound,defensive
4,foul,personal
6,rebound,offensive
41,substitution,out
42,substitution,in
...,...,...
2418867,made shot,Running Reverse Layup Shot
2419089,missed shot,Turnaround Hook Shot
2420121,instant replay,Coach Challenge Overturn Ruling
2420174,missed shot,Reverse Layup Shot


In [24]:
# Remove all "delay-of-game" and "game end" instances from the DataFrame
pbp_all = pbp_all[(pbp_all['subType'] != 'delay-of-game') & (pbp_all['subType'] != 'end')].reset_index(drop=True)


In [25]:
# Get the indices of all technical fouls
tech_foul_indices = pbp_all.index[pbp_all['subType'] == 'technical'].tolist()

# For each, get indices for 3 rows before and after (with bounds checks)
context_indices = set()
for idx in tech_foul_indices:
    for offset in range(-3, 4):
        check_idx = idx + offset
        if 0 <= check_idx < len(pbp_all):
            context_indices.add(check_idx)

# Sort indices for display
context_indices = sorted(context_indices)

# Show the rows
pbp_all.iloc[context_indices].head(18)

,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,periodNumber,quarter,periodSecondsLeft,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,actionType,subType,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,value,shortFormattedClock,jerseyNumber,location,videoAvailable,shotValue,actionId,line,homeFouls,awayFouls,homeFoulsAtCheckpoint,awayFoulsAtCheckpoint,homeNonPenaltyFoulLimit,awayNonPenaltyFoulLimit,homeBonus,awayBonus,isFlagrantFoul,isFlagrantFreeThrow,flagrantPenalty,flagrantFreeThrowsAwarded,flagrantCountsAsTeamFoul,flagrantCountsAsPersonalFoul,flagrantPossessionRetained,flagrantPossessionTeamSide,flagrantPlayerDisqualified,homeFreeThrows,awayFreeThrows,isFoulOut,isEjection,isTechnicalEjection,isDisqualificationEvent,disqualificationReason,disqualifiedPlayerPersonId,disqualifiedPlayerName,disqualifiedPlayerSide,homePlayersFouledOut,awayPlayersFouledOut,homePlayersEjected,awayPlayersEjected,homePlayersDisqualified,awayPlayersDisqualified,homeEjections,awayEjections
926,2021-22,0022100002,2021-10-19,LAL vs. GSW,1610612747,LAL,1610612744,GSW,0,D. Green REBOUND (Off:2 Def:2),3,Q3,632.0,1352.0,61.0,54.0,7.0,115.0,away,0.000,rebound,defensive,203110,Green,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.5,2,1,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
927,2021-22,0022100002,2021-10-19,LAL vs. GSW,1610612747,LAL,1610612744,GSW,0,NaN,3,Q3,626.0,1346.0,61.0,54.0,7.0,115.0,<NA>,0.000,stoppage,out-of-bounds,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.5,2,1,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
928,2021-22,0022100002,2021-10-19,LAL vs. GSW,1610612747,LAL,1610612744,GSW,0,R. Westbrook shooting personal FOUL (3 PF) (Gr...,3,Q3,623.0,1343.0,61.0,54.0,7.0,115.0,home,0.000,foul,personal,201566,Westbrook,NaN,3.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.5,3,1,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,2,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
929,2021-22,0022100002,2021-10-19,LAL vs. GSW,1610612747,LAL,1610612744,GSW,0,R. Westbrook technical FOUL (1 Tech),3,Q3,623.0,1343.0,61.0,54.0,7.0,115.0,home,0.000,foul,technical,201566,Westbrook,NaN,3.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.5,3,1,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,1,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
930,2021-22,0022100002,2021-10-19,LAL vs. GSW,1610612747,LAL,1610612744,GSW,0,S. Curry technical Free Throw 1 of 1 (11 PTS),3,Q3,623.0,1343.0,61.0,55.0,6.0,116.0,away,0.000,freethrow,1 of 1,201939,Curry,Made,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.5,3,1,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
931,2021-22,0022100002,2021-10-19,LAL vs. GSW,1610612747,LAL,1610612744,GSW,0,MISS D. Green Free Throw 1 of 2,3,Q3,623.0,1343.0,61.0,55.0,6.0,116.0,away,0.955,freethrow,1 of 2,203110,Green,Missed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.5,3,1,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,1,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
932,2021-22,0022100002,2021-10-19,LAL vs. GSW,1610612747,LAL,1610612744,GSW,0,TEAM offensive REBOUND,3,Q3,623.0,1343.0,61.0,55.0,6.0,116.0,away,0.000,rebound,offensive,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.5,3,1,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,1,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
1152,2021-22,0022100002,2021-10-19,LAL vs. GSW,1610612747,LAL,1610612744,GSW,0,L. James shooting personal FOUL (4 PF) (Bjelic...,4,Q4,249.0,249.0,101.0,108.0,-7.0,209.0,home,0.000,foul,personal,2544,James,NaN,4.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.5,3,2,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,2,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
1153,2021-22,0022100002,2021-10-19,LAL vs. GSW,1610612747,LAL,1610612744,GSW,0,N. Bjelica Free Throw 1 of 2 (14 PTS),4,Q4,249.0,249.0,101.0,109.0,-8.0,210.0,away,0.955,freethrow,1 of 2,202357,Bjelica,Made,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.5

Ejection/player-disqualification scratchpad

Simple disqualification/ejection model is now implemented in `applyPlayerDisqualificationFeatures`. Remaining optional enhancement: weight disqualified players by minutes, usage, or another player-value proxy.


In [26]:

tech_fouls = pbp_all[
    pbp_all["subType"] == "technical"
]

tech_foul_cols = [
    "gameId", "periodNumber", "quarter", "periodSecondsLeft", "description",
    "actionType", "subType",
    "homeFreeThrows", "awayFreeThrows",
    "actionTeamSide", "possession"
]

tech_fouls[[c for c in tech_foul_cols if c in tech_fouls.columns]].head(20)


,gameId,periodNumber,quarter,periodSecondsLeft,description,actionType,subType,homeFreeThrows,awayFreeThrows,actionTeamSide,possession
929,0022100002,3,Q3,623.0,R. Westbrook technical FOUL (1 Tech),foul,technical,0,1,home,0.0
1155,0022100002,4,Q4,237.0,N. Bjelica defensive-3-second technical FOUL (...,foul,technical,1,0,away,1.0
1947,0022100004,1,Q1,165.0,N. Vucevic technical FOUL (1 Tech),foul,technical,1,0,away,1.0
2407,0022100005,1,Q1,585.0,M. Robinson defensive-3-second technical FOUL ...,foul,technical,0,1,home,0.0
3281,0022100006,2,Q2,488.0,TEAM foul technical,foul,technical,0,1,home,0.0
3350,0022100006,2,Q2,222.0,M. Harrell technical FOUL (1 Tech),foul,technical,1,0,away,1.0
3917,0022100007,2,Q2,280.0,D. Bane technical FOUL (1 Tech),foul,technical,0,1,home,0.0
4832,0022100008,4,Q4,280.0,TEAM foul technical,foul,technical,1,1,home,1.0
5213,0022100009,3,Q3,270.0,J. Embiid technical FOUL (1 Tech),foul,technical,1,0,away,1.0
6256,0022100011,2,Q2,67.0,D. Mitchell technical FOUL (1 Tech),foul,technical,0,1,home,0.0


In [27]:
pbp_all.head()

,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,periodNumber,quarter,periodSecondsLeft,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,actionType,subType,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,value,shortFormattedClock,jerseyNumber,location,videoAvailable,shotValue,actionId,line,homeFouls,awayFouls,homeFoulsAtCheckpoint,awayFoulsAtCheckpoint,homeNonPenaltyFoulLimit,awayNonPenaltyFoulLimit,homeBonus,awayBonus,isFlagrantFoul,isFlagrantFreeThrow,flagrantPenalty,flagrantFreeThrowsAwarded,flagrantCountsAsTeamFoul,flagrantCountsAsPersonalFoul,flagrantPossessionRetained,flagrantPossessionTeamSide,flagrantPlayerDisqualified,homeFreeThrows,awayFreeThrows,isFoulOut,isEjection,isTechnicalEjection,isDisqualificationEvent,disqualificationReason,disqualifiedPlayerPersonId,disqualifiedPlayerName,disqualifiedPlayerSide,homePlayersFouledOut,awayPlayersFouledOut,homePlayersEjected,awayPlayersEjected,homePlayersDisqualified,awayPlayersDisqualified,homeEjections,awayEjections
0,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Period Start,1,Q1,720.0,2880.0,0.0,0.0,0.0,0.0,<NA>,0.5,period,start,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
1,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Jump Ball B. Lopez vs. N. Claxton: Tip to G. A...,1,Q1,715.0,2875.0,0.0,0.0,0.0,0.0,home,1.0,jumpball,recovered,203507,Antetokounmpo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
2,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,MISS G. Allen 26' 3PT,1,Q1,702.0,2862.0,0.0,0.0,0.0,0.0,home,0.3,3pt,Jump Shot,1628960,Allen,Missed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
3,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,K. Durant REBOUND (Off:0 Def:1),1,Q1,699.0,2859.0,0.0,0.0,0.0,0.0,away,0.0,rebound,defensive,201142,Durant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0
4,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,G. Antetokounmpo shooting personal FOUL (1 PF)...,1,Q1,687.0,2847.0,0.0,0.0,0.0,0.0,home,0.0,foul,personal,203507,Antetokounmpo,NaN,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,1,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,2,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0,0,0,0,0,0,0


In [28]:
pbp_all.drop(columns = ['homePlayersFouledOut',	"awayPlayersFouledOut",	"homePlayersEjected",	"awayPlayersEjected",	"homePlayersDisqualified",	"awayPlayersDisqualified"], inplace=True)

In [29]:
pbp_all.head()

,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,periodNumber,quarter,periodSecondsLeft,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,actionType,subType,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,value,shortFormattedClock,jerseyNumber,location,videoAvailable,shotValue,actionId,line,homeFouls,awayFouls,homeFoulsAtCheckpoint,awayFoulsAtCheckpoint,homeNonPenaltyFoulLimit,awayNonPenaltyFoulLimit,homeBonus,awayBonus,isFlagrantFoul,isFlagrantFreeThrow,flagrantPenalty,flagrantFreeThrowsAwarded,flagrantCountsAsTeamFoul,flagrantCountsAsPersonalFoul,flagrantPossessionRetained,flagrantPossessionTeamSide,flagrantPlayerDisqualified,homeFreeThrows,awayFreeThrows,isFoulOut,isEjection,isTechnicalEjection,isDisqualificationEvent,disqualificationReason,disqualifiedPlayerPersonId,disqualifiedPlayerName,disqualifiedPlayerSide,homeEjections,awayEjections
0,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Period Start,1,Q1,720.0,2880.0,0.0,0.0,0.0,0.0,<NA>,0.5,period,start,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0
1,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Jump Ball B. Lopez vs. N. Claxton: Tip to G. A...,1,Q1,715.0,2875.0,0.0,0.0,0.0,0.0,home,1.0,jumpball,recovered,203507,Antetokounmpo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0
2,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,MISS G. Allen 26' 3PT,1,Q1,702.0,2862.0,0.0,0.0,0.0,0.0,home,0.3,3pt,Jump Shot,1628960,Allen,Missed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0
3,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,K. Durant REBOUND (Off:0 Def:1),1,Q1,699.0,2859.0,0.0,0.0,0.0,0.0,away,0.0,rebound,defensive,201142,Durant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,0,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0
4,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,G. Antetokounmpo shooting personal FOUL (1 PF)...,1,Q1,687.0,2847.0,0.0,0.0,0.0,0.0,home,0.0,foul,personal,203507,Antetokounmpo,NaN,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,1,0,<NA>,<NA>,4,4,0,0,0,0,<NA>,<NA>,0,0,0,<NA>,0,0,2,0,0,0,0,<NA>,<NA>,<NA>,<NA>,0,0


In [30]:
pbp_all.drop(columns=['subType', 'personId', 'playerName', 'foulPersonalTotal', 'foulTechnicalTotal', 'homeFoulsAtCheckpoint',	"awayFoulsAtCheckpoint",	"homeNonPenaltyFoulLimit",	"awayNonPenaltyFoulLimit",
"isFlagrantFoul",	"isFlagrantFreeThrow",	"flagrantPenalty",	"flagrantFreeThrowsAwarded",	"flagrantCountsAsTeamFoul",	"flagrantCountsAsPersonalFoul",	"flagrantPossessionRetained",	"flagrantPossessionTeamSide",	"flagrantPlayerDisqualified",
"isFoulOut",	"isEjection",	"isTechnicalEjection",	"isDisqualificationEvent",	"disqualificationReason",	"disqualifiedPlayerPersonId",	"disqualifiedPlayerName",	"disqualifiedPlayerSide"], inplace=True)

In [31]:
pbp_all.drop(columns=['periodNumber'], inplace=True)

In [32]:
pbp_all.head(3)

,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,quarter,periodSecondsLeft,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,actionType,shotResult,value,shortFormattedClock,jerseyNumber,location,videoAvailable,shotValue,actionId,line,homeFouls,awayFouls,homeBonus,awayBonus,homeFreeThrows,awayFreeThrows,homeEjections,awayEjections
0,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Period Start,Q1,720.0,2880.0,0.0,0.0,0.0,0.0,<NA>,0.5,period,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,0,0,0,0,0,0
1,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Jump Ball B. Lopez vs. N. Claxton: Tip to G. A...,Q1,715.0,2875.0,0.0,0.0,0.0,0.0,home,1.0,jumpball,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,0,0,0,0,0,0
2,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,MISS G. Allen 26' 3PT,Q1,702.0,2862.0,0.0,0.0,0.0,0.0,home,0.3,3pt,Missed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,0,0,0,0,0,0


### Note: Don't use the team scores for modeling

In [33]:
# Fixing possession column
mask = (
    ((pbp_all['actionType'] == '3pt') | (pbp_all['actionType'] == '2pt')) &
    (pbp_all['shotResult'] == 'Made')
)

pbp_all.loc[mask & (pbp_all['actionTeamSide'] == 'away'), 'possession'] = 1
pbp_all.loc[mask & (pbp_all['actionTeamSide'] == 'home'), 'possession'] = 0

In [34]:
# Drop all substitution and timeout rows
sub_mask = pbp_all['actionType'] != 'substitution'
timeout_mask = pbp_all['actionType'] != 'timeout'
pbp_all = pbp_all[sub_mask & timeout_mask]
pbp_all.head()

,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,quarter,periodSecondsLeft,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,actionType,shotResult,value,shortFormattedClock,jerseyNumber,location,videoAvailable,shotValue,actionId,line,homeFouls,awayFouls,homeBonus,awayBonus,homeFreeThrows,awayFreeThrows,homeEjections,awayEjections
0,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Period Start,Q1,720.0,2880.0,0.0,0.0,0.0,0.0,<NA>,0.5,period,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,0,0,0,0,0,0
1,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Jump Ball B. Lopez vs. N. Claxton: Tip to G. A...,Q1,715.0,2875.0,0.0,0.0,0.0,0.0,home,1.0,jumpball,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,0,0,0,0,0,0
2,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,MISS G. Allen 26' 3PT,Q1,702.0,2862.0,0.0,0.0,0.0,0.0,home,0.3,3pt,Missed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,0,0,0,0,0,0
3,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,K. Durant REBOUND (Off:0 Def:1),Q1,699.0,2859.0,0.0,0.0,0.0,0.0,away,0.0,rebound,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,0,0,0,0,0,0,0,0
4,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,G. Antetokounmpo shooting personal FOUL (1 PF)...,Q1,687.0,2847.0,0.0,0.0,0.0,0.0,home,0.0,foul,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.5,1,0,0,0,0,2,0,0


In [37]:
pbp_all.drop(columns = ['value', 'shortFormattedClock', 'jerseyNumber', 'location',	'videoAvailable', 'shotValue', 'actionId', 'shotResult'], inplace=True)

In [38]:
pbp_all.head()

,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,quarter,periodSecondsLeft,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,actionType,line,homeFouls,awayFouls,homeBonus,awayBonus,homeFreeThrows,awayFreeThrows,homeEjections,awayEjections
0,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Period Start,Q1,720.0,2880.0,0.0,0.0,0.0,0.0,<NA>,0.5,period,1.5,0,0,0,0,0,0,0,0
1,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,Jump Ball B. Lopez vs. N. Claxton: Tip to G. A...,Q1,715.0,2875.0,0.0,0.0,0.0,0.0,home,1.0,jumpball,1.5,0,0,0,0,0,0,0,0
2,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,MISS G. Allen 26' 3PT,Q1,702.0,2862.0,0.0,0.0,0.0,0.0,home,0.3,3pt,1.5,0,0,0,0,0,0,0,0
3,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,K. Durant REBOUND (Off:0 Def:1),Q1,699.0,2859.0,0.0,0.0,0.0,0.0,away,0.0,rebound,1.5,0,0,0,0,0,0,0,0
4,2021-22,0022100001,2021-10-19,MIL vs. BKN,1610612749,MIL,1610612751,BKN,1,G. Antetokounmpo shooting personal FOUL (1 PF)...,Q1,687.0,2847.0,0.0,0.0,0.0,0.0,home,0.0,foul,1.5,1,0,0,0,0,2,0,0


In [39]:
pbp_all.to_csv("pbp_all.csv")